<a href="https://colab.research.google.com/github/varadbarclays/dev1/blob/main/Agentic_Retrieval_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ACM ICAIF-25 AI Agentic Retrieval Grand Challenge

---

## Overview

The competition consists of two main ranking tasks:

1. **Document Ranking** – Identify and rank the five most relevant documents.  
2. **Chunk Ranking** – Identify and rank the five most relevant text chunks.

---

## Processing Pipeline

- **Data Loading**  
  Reads evaluation data from JSONL files.

- **Token Analysis**  
  Checks input size to decide whether it exceed the context length of the model..

- **Smart Ranking**  
  - *Normal cases*: Single-stage ranking.  
  - *High-token cases*: Multi-stage divide-and-conquer ranking to handle large inputs efficiently.

- **Result Compilation**  
  Combines model outputs and prepares a final CSV submission file.

---

## Output

- **kaggle_submission.csv**  
  Ready-to-submit file containing the required `sample_id` and `target_index` columns.

- **Comprehensive Statistics**  
  Summarized metrics and analysis of ranking results.

- **Top-5 Rankings**  
  Returns the five most relevant items for each query as required by the challenge.

---

## Usage

This notebook enables **end-to-end evaluation**: from loading data to generating a Kaggle-ready submission file.  
Simply run the pipeline and upload the generated `kaggle_submission.csv` to the competition platform.

In [ ]:
!pip install openai tiktoken python-dotenv pydantic tqdm

In [ ]:
import pandas as pd
import json
import ast
import re

import asyncio
import csv
import json
import os
import traceback
from typing import Dict, List

import tiktoken
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel
from tqdm.asyncio import tqdm

load_dotenv()

False

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"varadsrivastava","key":"07be8d5799432154520a4aff77f26a0d"}'}

In [ ]:
!ls -lha kaggle.json
!pip install -q kaggle # installing the kaggle package
!mkdir -p ~/.kaggle # creating .kaggle folder where the key should be placed
!cp kaggle.json ~/.kaggle/ # move the key to the folder
!pwd # checking the present working directory

-rw-r--r-- 1 root root 71 Sep 29 15:25 kaggle.json
/content


In [ ]:
# giving rw access (if 401-nathorized)

# !chmod 600 ~/.kaggle/kaggle.json

# Dataset

In [ ]:
!kaggle competitions download -c acm-icaif-25-ai-agentic-retrieval-grand-challenge

 93% 1.14G/1.23G [00:06<00:00, 102MB/s] 
100% 1.23G/1.23G [00:06<00:00, 199MB/s]


In [ ]:
!unzip *acm-icaif-25-ai-agentic-retrieval-grand-challenge.zip

Archive:  acm-icaif-25-ai-agentic-retrieval-grand-challenge.zip
  inflating: chunk_ranking_kaggle_dev.jsonl  
  inflating: chunk_ranking_kaggle_eval.jsonl  
  inflating: document_ranking_kaggle_dev.jsonl  
  inflating: document_ranking_kaggle_eval.jsonl  
  inflating: kaggle_submission.csv   


# Documents

In [ ]:
file_path = '/content/document_ranking_kaggle_dev.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_doc_dev = pd.DataFrame(data)
print(f"Head of {file_path}:")
display(df_doc_dev.head())

Head of /content/document_ranking_kaggle_dev.jsonl:


,uuid,messages,qrel
0,qe100cdf8e8f5,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}"
1,q723817294fbe,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '0': 2, '1': 1, '3': 1, '2': 0}"
2,qd79970afaa57,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '1': 2, '0': 1, '2': 1, '3': 0}"
3,qeb144d0e9991,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 3, '2': 2, '4': 1, '0': 0, '3': 0}"
4,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '2': 2, '1': 1, '0': 0, '3': 0}"


In [ ]:
df_doc_dev

,uuid,messages,qrel
0,qe100cdf8e8f5,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}"
1,q723817294fbe,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '0': 2, '1': 1, '3': 1, '2': 0}"
2,qd79970afaa57,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '1': 2, '0': 1, '2': 1, '3': 0}"
3,qeb144d0e9991,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 3, '2': 2, '4': 1, '0': 0, '3': 0}"
4,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '2': 2, '1': 1, '0': 0, '3': 0}"
...,...,...,...
4981,q30f7c8056c93,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 4, '2': 3, '0': 2, '4': 1, '3': 0}"
4982,q402c59dfe9b1,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 4, '0': 3, '4': 2, '3': 1, '2': 0}"
4983,q45a9a878195e,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 4, '2': 3, '1': 2, '0': 1, '3': 0}"
4984,q2c57b56ff808,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 4, '1': 3, '2': 2, '0': 1, '3': 0}"


In [ ]:
df_doc_dev["messages"][0]

[{'role': 'user',
  'content': 'Rank the following financial document types by relevance to answer the question. Provide your ranking as a list of indices from most relevant to least relevant.\n\nQuestion: How has Agilent Technologies’ instrument reliability metric for its core diagnostics manufacturing process changed recently?\n\nDocument Types to rank:\n[Document Index 0] DEF14A\n\n[Document Index 1] 10-K\n\n[Document Index 2] 10-Q\n\n[Document Index 3] 8-K\n\n[Document Index 4] Earnings\n\nYour response must be a list of indices in exact list format (e.g., [4, 2, 1, 0, 3]), ranking every index from 0 to 4 by most relevant document type index to least relevant document type index.'}]

In [ ]:
file_path = '/content/document_ranking_kaggle_eval.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_doc_eval = pd.DataFrame(data)
print(f"Head of {file_path}:")
display(df_doc_eval.head())

Head of /content/document_ranking_kaggle_eval.jsonl:


,_id,messages
0,doc_q39d7b7,"[{'role': 'user', 'content': 'Rank the followi..."
1,doc_q8edbb8,"[{'role': 'user', 'content': 'Rank the followi..."
2,doc_q060a50,"[{'role': 'user', 'content': 'Rank the followi..."
3,doc_q3ec868,"[{'role': 'user', 'content': 'Rank the followi..."
4,doc_qc7db20,"[{'role': 'user', 'content': 'Rank the followi..."


In [ ]:
# extract the question from df_doc_dev between "Question: " to "\n\nDocument"
df_doc_dev["question"] = df_doc_dev["messages"].apply(lambda x: x[0]["content"].split("Question: ")[1].split("\n\nDocument")[0])

df_doc_eval["question"] = df_doc_eval["messages"].apply(lambda x: x[0]["content"].split("Question: ")[1].split("\n\nDocument")[0])


In [ ]:
df_doc_dev["question"][2]

'How do sustainability or ESG considerations influence customer demand in Agilent Technologies’ market?'

In [ ]:
df_doc_eval["question"][0]

'How has Salesforce’s subscription and support segment profitability trended over recent periods?'

# Chunks

In [ ]:
file_path = '/content/chunk_ranking_kaggle_dev.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_chunk_dev = pd.DataFrame(data)
print(f"Head of {file_path}:")
display(df_chunk_dev.head())

Head of /content/chunk_ranking_kaggle_dev.jsonl:


,uuid,messages,qrel
0,q7d7a32439929,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': ..."
1,qeb144d0e9991,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
2,qd79970afaa57,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 2, '2': 0, '3': 0, '4': 0, '5': ..."
3,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
4,qc640e7d1c938,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."


In [ ]:
df_chunk_dev

,uuid,messages,qrel
0,q7d7a32439929,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': ..."
1,qeb144d0e9991,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
2,qd79970afaa57,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 2, '2': 0, '3': 0, '4': 0, '5': ..."
3,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
4,qc640e7d1c938,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
...,...,...,...
18850,q5dcbc1a08e24,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 1, '1': 2, '2': 0, '3': 0, '4': 0, '5': ..."
18851,qe231f9f4a31c,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."
18852,q402c59dfe9b1,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': ..."
18853,q4342a9af31f9,"[{'role': 'user', 'content': 'Identify the 10 ...","{'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': ..."


In [ ]:
df_chunk_dev["qrel"][0]

{'0': 0,
 '1': 0,
 '2': 0,
 '3': 0,
 '4': 1,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 2,
 '9': 0,
 '10': 0,
 '11': 0,
 '12': 0,
 '13': 0,
 '14': 0,
 '15': 0,
 '16': 0,
 '17': 0,
 '18': 0,
 '19': 0,
 '20': 0,
 '21': 0,
 '22': 0,
 '23': 0,
 '24': 2,
 '25': 0,
 '26': 0,
 '27': 0,
 '28': 0,
 '29': 0,
 '30': 0,
 '31': 0,
 '32': 0,
 '33': 0,
 '34': 0,
 '35': 0,
 '36': 0,
 '37': 0,
 '38': 0,
 '39': 0,
 '40': 0,
 '41': 0,
 '42': 1,
 '43': 0,
 '44': 0,
 '45': 0,
 '46': 0,
 '47': 0,
 '48': 0,
 '49': 0,
 '50': 0,
 '51': 0,
 '52': 0,
 '53': 0,
 '54': 0,
 '55': 0,
 '56': 0,
 '57': 0,
 '58': 0,
 '59': 0,
 '60': 0,
 '61': 0,
 '62': 0,
 '63': 0,
 '64': 2,
 '65': 0,
 '66': 0,
 '67': 0,
 '68': 0,
 '69': 0,
 '70': 0,
 '71': 0,
 '72': 0,
 '73': 0,
 '74': 0,
 '75': 0,
 '76': 0,
 '77': 0,
 '78': 0,
 '79': 0,
 '80': 0,
 '81': 0,
 '82': 0,
 '83': 0,
 '84': 0,
 '85': 0,
 '86': 0,
 '87': 0,
 '88': 0,
 '89': 0,
 '90': 0,
 '91': 0,
 '92': 0,
 '93': 0,
 '94': 0,
 '95': 0,
 '96': 0,
 '97': 0,
 '98': 0,
 '99': 0,
 '100': 0,

In [ ]:
file_path = '/content/chunk_ranking_kaggle_eval.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_chunk_eval = pd.DataFrame(data)
print(f"Head of {file_path}:")
display(df_chunk_eval.head())

Head of /content/chunk_ranking_kaggle_eval.jsonl:


,_id,messages
0,chunk_q613509,"[{'role': 'user', 'content': 'Identify the 10 ..."
1,chunk_q83d560,"[{'role': 'user', 'content': 'Identify the 10 ..."
2,chunk_qc06926,"[{'role': 'user', 'content': 'Identify the 10 ..."
3,chunk_qaae0f2,"[{'role': 'user', 'content': 'Identify the 10 ..."
4,chunk_q059068,"[{'role': 'user', 'content': 'Identify the 10 ..."


In [ ]:
# extract the chunks from df_chunk_dev between "(best first).\n" to "\n- Put the BEST chunk"
df_chunk_dev["chunks"] = df_chunk_dev["messages"].apply(lambda x: x[0]["content"].split("(best first).\n")[1].split("\n\nTask: Select and rank")[0])

df_chunk_eval["chunks"] = df_chunk_eval["messages"].apply(lambda x: x[0]["content"].split("(best first).\n")[1].split("\n\nTask: Select and rank")[0])


In [ ]:
len(df_chunk_dev["messages"][0][0]["content"])

298154

In [ ]:
len(df_chunk_dev["chunks"][0])

297719

In [ ]:
df_chunk_dev["chunks"][0]

'Question: What share ownership guidelines are defined for Agilent Technologies’ executives and directors?\nText chunks:\n[Chunk Index 0] # UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n\n# SCHEDULE 14A INFORMATION\n\n# PROXY STATEMENT PURSUANT TO SECTION 14(a) OF THE\nSECURITIES EXCHANGE ACT OF 1934\n(AMENDMENT NO. )\nSCHEDULE 14A\n\nFiled by:\n[x] Filed by the Registrant\n[ ] Filed by a Party other than the Registrant\n\nCheck the appropriate box:\n[ ] Preliminary Proxy Statement\n[ ] Confidential, for Use of the Commission Only (as permitted by Rule 14a-6(e)(2))\n[x] Definitive Proxy Statement\n[ ] Definitive Additional Materials\n[ ] Soliciting Material Pursuant to §240.14a-12\n\n## AGILENT TECHNOLOGIES, INC.\n\n(Name of Registrant as Specified In Its Charter)\n\n(Name of Person(s) Filing Proxy Statement, if other than the Registrant)\n\nPayment of Filing Fee (Check the appropriate box):\n\n- No fee required.\n\n- Fee paid previously with preliminary ma

In [ ]:
df_chunk_dev["messages"][0]

[{'role': 'user',
  'content': 'Identify the 10 most relevant text chunks for answering this question, then rank them in order of relevance (best first).\nQuestion: What share ownership guidelines are defined for Agilent Technologies’ executives and directors?\nText chunks:\n[Chunk Index 0] # UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n\n# SCHEDULE 14A INFORMATION\n\n# PROXY STATEMENT PURSUANT TO SECTION 14(a) OF THE\nSECURITIES EXCHANGE ACT OF 1934\n(AMENDMENT NO. )\nSCHEDULE 14A\n\nFiled by:\n[x] Filed by the Registrant\n[ ] Filed by a Party other than the Registrant\n\nCheck the appropriate box:\n[ ] Preliminary Proxy Statement\n[ ] Confidential, for Use of the Commission Only (as permitted by Rule 14a-6(e)(2))\n[x] Definitive Proxy Statement\n[ ] Definitive Additional Materials\n[ ] Soliciting Material Pursuant to §240.14a-12\n\n## AGILENT TECHNOLOGIES, INC.\n\n(Name of Registrant as Specified In Its Charter)\n\n(Name of Person(s) Filing Proxy Stateme

In [ ]:
# df_chunk_eval["chunks"][5]

# Trying embedding route for Documents and DeepSeek for TRAC based rank of Chunks

## Create instance of LLM

In [ ]:
# Setup DeepSeek LLM
from google.colab import userdata
# Use DeepSeek Model

try:
    DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
except Exception as e:
    print(f"Error configuring DeepSeek API: {e}")
    print("Please make sure you have added your API key to Colab secrets as 'DEEPSEEK_API_KEY'")

In [ ]:
from openai import OpenAI

In [ ]:
def deepseek_model(system_prompt, user_prompt):
    """Generates Question-Thought-Answer triplets from the given text using OpenAI."""
    try:
        client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}
        )

        # print(response)

        # Attempt to parse the response as JSON
        try:
            rankings = json.loads(response.choices[0].message.content)
            return rankings
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from model response: {e}")
            print("Raw response text:", response.choices[0].message.content)
            return []

    except Exception as e:
        print(f"An error occurred during ranking generation: {e}")
        return []

## Get embeddings

In [ ]:
# document is of form:
# question:
# ranks of document types based on relevancy of document for the ques
# but the ranks need not be 4,3,2,1,0, sometimes they are 1,0,0,0,0 meaning

# use MPNetv2 based embedding based RAG, which passes the five most similar
# questions and their ranks to the LLM


# load sentence transformer MPNetv2

from sentence_transformers import SentenceTransformer
mpnet = SentenceTransformer('all-mpnet-base-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Extract questions from development and evaluation dataframes
dev_questions = df_doc_dev["question"].tolist()
eval_questions = df_doc_eval["question"].tolist()

# Generate embeddings for development questions
dev_embeddings = mpnet.encode(dev_questions)

# Generate embeddings for evaluation questions
eval_embeddings = mpnet.encode(eval_questions)

print("Embeddings generated for development and evaluation questions.")

Embeddings generated for development and evaluation questions.


In [ ]:
df_doc_dev["qrel"][0]

{'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}

## Cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity between evaluation and development embeddings
similarity_scores = cosine_similarity(eval_embeddings, dev_embeddings)

print("Cosine similarity scores calculated.")

Cosine similarity scores calculated.


In [ ]:
import numpy as np

top_k = 5
top_k_qrels = []
top_k_ques = []

for scores in similarity_scores:
    # Get indices of top k similarity scores in descending order
    top_k_indices = np.argsort(scores)[::-1][:top_k]

    # Retrieve the qrel for the top k development questions
    questions_for_eval_q = df_doc_dev.iloc[top_k_indices]['question'].tolist()
    qrels_for_eval_q = df_doc_dev.iloc[top_k_indices]['qrel'].tolist()

    top_k_ques.append(questions_for_eval_q)
    top_k_qrels.append(qrels_for_eval_q)

# Add the top k qrels to the evaluation dataframe
df_doc_eval['top_k_dev_ques'] = top_k_ques
df_doc_eval['top_k_dev_qrels'] = top_k_qrels


print("Top 5 development question qrels retrieved and added to df_doc_eval.")
display(df_doc_eval.head())

Top 5 development question qrels retrieved and added to df_doc_eval.


,_id,messages,question,top_k_dev_ques,top_k_dev_qrels
0,doc_q39d7b7,"[{'role': 'user', 'content': 'Rank the followi...",How has Salesforce’s subscription and support ...,[How has PTC Inc.’s software subscription segm...,"[{'2': 4, '1': 3, '3': 2, '4': 1, '0': 0}, {'1..."
1,doc_q8edbb8,"[{'role': 'user', 'content': 'Rank the followi...",How does Blackstone manage equity award burn r...,"[How does Brown & Brown, Inc. manage equity aw...","[{'1': 3, '0': 2, '2': 1, '3': 0, '4': 0}, {'0..."
2,doc_q060a50,"[{'role': 'user', 'content': 'Rank the followi...",What questions were asked about A. O. Smith Co...,[What questions were asked about Stanley Black...,"[{'4': 4, '3': 3, '2': 2, '0': 1, '1': 0}, {'4..."
3,doc_q3ec868,"[{'role': 'user', 'content': 'Rank the followi...","How has the ratio of Corpay, Inc.’s recurring ...",[How has the ratio of Newmont Corporation’s re...,"[{'2': 4, '1': 3, '3': 2, '4': 1, '0': 0}, {'1..."
4,doc_qc7db20,"[{'role': 'user', 'content': 'Rank the followi...",What questions were asked about Ford Motor Com...,[What questions were asked about General Motor...,"[{'4': 4, '0': 3, '3': 2, '2': 1, '1': 0}, {'4..."


In [ ]:
df_doc_eval['top_k_dev_ques'][0]

['How has PTC Inc.’s software subscription segment profitability trended over recent periods?',
 'How has Oracle’s cloud services segment profitability trended over recent periods?',
 'How has Alphabet Inc.’s Google Cloud segment profitability trended over recent periods?',
 'How has Amazon’s AWS segment profitability trended over recent periods?',
 'How has Skyworks Solutions, Inc.’s mobile solutions segment profitability trended over recent periods?']

In [ ]:
df_doc_eval['top_k_dev_qrels'][0]

[{'2': 4, '1': 3, '3': 2, '4': 1, '0': 0},
 {'1': 4, '2': 3, '3': 2, '4': 1, '0': 0},
 {'1': 4, '2': 3, '3': 2, '4': 1, '0': 0},
 {'1': 4, '2': 3, '3': 2, '4': 1, '0': 0},
 {'4': 4, '1': 3, '2': 2, '0': 1, '3': 0}]

In [ ]:
# convert keys in qrels into strings for easier understanding of LLM
docno_to_name = {"0": "DEF14A", "1": "10-K", "2": "10-Q", "3": "8-K", "4": "Earnings"}


In [ ]:
# convert doc string keys in the df_doc_eval['top_k_dev_qrels'] using docno_to_nam
for i in range(len(df_doc_eval['top_k_dev_qrels'])):
  temp = df_doc_eval['top_k_dev_qrels'][i]
  df_doc_eval['top_k_dev_qrels'][i] = [{docno_to_name[k]: v for k, v in j.items()} for j in temp]

/tmp/ipython-input-1748343892.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_doc_eval['top_k_dev_qrels'][i] = [{docno_to_name[k]: v for k, v in j.items()} for j in temp]
/tmp/ipython-input-1748343892.py:4: FutureWarning: ChainedAssig

In [ ]:
  df_doc_eval['top_k_dev_qrels'][0]

[{'10-Q': 4, '10-K': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0},
 {'10-K': 4, '10-Q': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0},
 {'10-K': 4, '10-Q': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0},
 {'10-K': 4, '10-Q': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0},
 {'Earnings': 4, '10-K': 3, '10-Q': 2, 'DEF14A': 1, '8-K': 0}]

In [ ]:
df_doc_dev["messages"][0][0]["content"]

'Rank the following financial document types by relevance to answer the question. Provide your ranking as a list of indices from most relevant to least relevant.\n\nQuestion: How has Agilent Technologies’ instrument reliability metric for its core diagnostics manufacturing process changed recently?\n\nDocument Types to rank:\n[Document Index 0] DEF14A\n\n[Document Index 1] 10-K\n\n[Document Index 2] 10-Q\n\n[Document Index 3] 8-K\n\n[Document Index 4] Earnings\n\nYour response must be a list of indices in exact list format (e.g., [4, 2, 1, 0, 3]), ranking every index from 0 to 4 by most relevant document type index to least relevant document type index.'

## Get deepseek ranking

### Subtask:
Use the DeepSeek model to process the formatted input and obtain a predicted ranking for the evaluation question.


In [ ]:
deepseek_prompts = []

system_prompt = """You are a helpful assistant that provides financial document rankings based on
which document is most likely to contain the information relevant to the question. Output only the ranking dictionary. Do not add any thinking, reasoning or other texts."""

for index, row in df_doc_eval.iterrows():
    eval_question = row['question']
    top_k_ques = row['top_k_dev_ques']
    top_k_qrels = row['top_k_dev_qrels']

    prompt = f"""Evaluate the relevance ranking of financial document types for the following question based on the provided examples.\n\nEvaluation Question: {eval_question}\n\n"""

    for i, qrel_dict in enumerate(top_k_qrels):
        example_question = top_k_ques[i]
        prompt += f"Example {i+1}:\n"
        prompt += f"Question: {example_question}\n"
        prompt += f"Rankings: {qrel_dict}\n\n"

    prompt += """Your Task:\nBased on the examples, rank the documents in the order of which document type is most likely to contain the answer based on the given question. Return the result ranking in a JSON format.\nAnswer Ranking:"""

    deepseek_prompts.append(prompt)

df_doc_eval['deepseek_prompts'] = deepseek_prompts

print("DeepSeek prompts generated and added to df_doc_eval.")
display(df_doc_eval.head())

DeepSeek prompts generated and added to df_doc_eval.


,_id,messages,question,top_k_dev_ques,top_k_dev_qrels,deepseek_prompts
0,doc_q39d7b7,"[{'role': 'user', 'content': 'Rank the followi...",How has Salesforce’s subscription and support ...,[How has PTC Inc.’s software subscription segm...,"[{'10-Q': 4, '10-K': 3, '8-K': 2, 'Earnings': ...",Evaluate the relevance ranking of financial do...
1,doc_q8edbb8,"[{'role': 'user', 'content': 'Rank the followi...",How does Blackstone manage equity award burn r...,"[How does Brown & Brown, Inc. manage equity aw...","[{'10-K': 3, 'DEF14A': 2, '10-Q': 1, '8-K': 0,...",Evaluate the relevance ranking of financial do...
2,doc_q060a50,"[{'role': 'user', 'content': 'Rank the followi...",What questions were asked about A. O. Smith Co...,[What questions were asked about Stanley Black...,"[{'Earnings': 4, '8-K': 3, '10-Q': 2, 'DEF14A'...",Evaluate the relevance ranking of financial do...
3,doc_q3ec868,"[{'role': 'user', 'content': 'Rank the followi...","How has the ratio of Corpay, Inc.’s recurring ...",[How has the ratio of Newmont Corporation’s re...,"[{'10-Q': 4, '10-K': 3, '8-K': 2, 'Earnings': ...",Evaluate the relevance ranking of financial do...
4,doc_qc7db20,"[{'role': 'user', 'content': 'Rank the followi...",What questions were asked about Ford Motor Com...,[What questions were asked about General Motor...,"[{'Earnings': 4, 'DEF14A': 3, '8-K': 2, '10-Q'...",Evaluate the relevance ranking of financial do...


In [ ]:
df_doc_eval["deepseek_prompts"][0]

"Evaluate the relevance ranking of financial document types for the following question based on the provided examples.\n\nEvaluation Question: How has Salesforce’s subscription and support segment profitability trended over recent periods?\n\nExample 1:\nQuestion: How has PTC Inc.’s software subscription segment profitability trended over recent periods?\nRankings: {'10-Q': 4, '10-K': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0}\n\nExample 2:\nQuestion: How has Oracle’s cloud services segment profitability trended over recent periods?\nRankings: {'10-K': 4, '10-Q': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0}\n\nExample 3:\nQuestion: How has Alphabet Inc.’s Google Cloud segment profitability trended over recent periods?\nRankings: {'10-K': 4, '10-Q': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0}\n\nExample 4:\nQuestion: How has Amazon’s AWS segment profitability trended over recent periods?\nRankings: {'10-K': 4, '10-Q': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0}\n\nExample 5:\nQuestion: How has Skyw

In [ ]:
ranking = deepseek_model(system_prompt, df_doc_eval["deepseek_prompts"][0])

In [ ]:
ranking

{'10-K': 4, '10-Q': 3, 'Earnings': 2, '8-K': 1, 'DEF14A': 0}

In [ ]:
rankings = []
for i in df_doc_eval["deepseek_prompts"]:
  ranking = deepseek_model(system_prompt, i)
  print(ranking)
  rankings.append(ranking)

df_doc_eval["rankings"] = rankings


{'10-K': 4, '10-Q': 3, 'Earnings': 2, '8-K': 1, 'DEF14A': 0}
{'10-K': 4, 'DEF14A': 3, '10-Q': 2, '8-K': 1, 'Earnings': 0}
{'Earnings': 4, '8-K': 3, '10-Q': 2, 'DEF14A': 1, '10-K': 0}
{'10-Q': 4, '10-K': 3, 'Earnings': 2, '8-K': 1, 'DEF14A': 0}
{'Earnings': 4, '8-K': 3, '10-Q': 2, 'DEF14A': 1, '10-K': 0}
{'10-K': 4, 'Earnings': 3, 'DEF14A': 2, '8-K': 1, '10-Q': 0}
{'10-K': 4, '10-Q': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0}
{'10-K': 4, 'DEF14A': 3, '10-Q': 2, '8-K': 1, 'Earnings': 0}
{'Earnings': 4, '10-Q': 3, '8-K': 2, '10-K': 1, 'DEF14A': 0}
{'10-K': 4, '10-Q': 3, 'Earnings': 2, 'DEF14A': 1, '8-K': 0}
{'10-K': 4, '10-Q': 3, 'Earnings': 2, 'DEF14A': 1, '8-K': 0}
{'Earnings': 4, 'DEF14A': 3, '8-K': 2, '10-Q': 1, '10-K': 0}
{'10-K': 4, '10-Q': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0}
{'Earnings': 4, '10-K': 3, '8-K': 2, 'DEF14A': 1, '10-Q': 0}
{'10-K': 4, 'DEF14A': 3, '10-Q': 2, '8-K': 1, 'Earnings': 0}
{'Earnings': 4, '10-K': 3, '10-Q': 2, 'DEF14A': 1, '8-K': 0}
{'10-Q': 4, '10-K': 3, '

In [ ]:
df_doc_eval['rankings']

,rankings
0,"{'10-K': 4, '10-Q': 3, 'Earnings': 2, '8-K': 1..."
1,"{'10-K': 4, 'DEF14A': 3, '10-Q': 2, '8-K': 1, ..."
2,"{'Earnings': 4, '8-K': 3, '10-Q': 2, 'DEF14A':..."
3,"{'10-Q': 4, '10-K': 3, 'Earnings': 2, '8-K': 1..."
4,"{'Earnings': 4, '8-K': 3, '10-Q': 2, 'DEF14A':..."
...,...
195,"{'DEF14A': 4, '10-K': 3, 'Earnings': 2, '8-K':..."
196,"{'10-K': 4, '10-Q': 3, 'Earnings': 2, '8-K': 1..."
197,"{'10-K': 4, '10-Q': 3, 'Earnings': 2, '8-K': 1..."
198,"{'10-K': 4, '10-Q': 3, '8-K': 2, 'Earnings': 1..."


In [ ]:
# convert ranking back to numbers
name_to_docno = {"DEF14A": "0", "10-K": "1", "10-Q": "2", "8-K": "3", "Earnings": "4"}

for i in range(len(df_doc_eval['rankings'])):
  temp = df_doc_eval['rankings'][i]
  df_doc_eval['rankings'][i] = [{name_to_docno[k]: v for k, v in temp.items()}]

In [ ]:
df_doc_eval['rankings']

,rankings
0,"[{'1': 4, '2': 3, '4': 2, '3': 1, '0': 0}]"
1,"[{'1': 4, '0': 3, '2': 2, '3': 1, '4': 0}]"
2,"[{'4': 4, '3': 3, '2': 2, '0': 1, '1': 0}]"
3,"[{'2': 4, '1': 3, '4': 2, '3': 1, '0': 0}]"
4,"[{'4': 4, '3': 3, '2': 2, '0': 1, '1': 0}]"
...,...
195,"[{'0': 4, '1': 3, '4': 2, '3': 1, '2': 0}]"
196,"[{'1': 4, '2': 3, '4': 2, '3': 1, '0': 0}]"
197,"[{'1': 4, '2': 3, '4': 2, '3': 1, '0': 0}]"
198,"[{'1': 4, '2': 3, '3': 2, '4': 1, '0': 0}]"


In [ ]:
# convert into submission form

#create empty df with columns
results = pd.DataFrame(columns=['sample_id', 'target_index'])

for index, i in enumerate(df_doc_eval['_id']):
  for j in range(0,5):
    current_id = i
    current_rankings = df_doc_eval['rankings'][index][0]

    # get key of the item in current ranking for which value is j
    for key, value in current_rankings.items():
      if value == j:
        doc_rank = {'sample_id': current_id, 'target_index': key}
        # add doc rank to df
        results = pd.concat([results, pd.DataFrame([doc_rank])], ignore_index=True)

In [ ]:
results

,sample_id,target_index
0,doc_q39d7b7,0
1,doc_q39d7b7,3
2,doc_q39d7b7,4
3,doc_q39d7b7,2
4,doc_q39d7b7,1
...,...,...
995,doc_qc6264f,0
996,doc_qc6264f,1
997,doc_qc6264f,2
998,doc_qc6264f,3


In [ ]:
# save results so far to csv
results.to_csv('results.csv', index=False)

## For chunks:
For annotating those relevance scores, we followed
TREC Eval: 0 (irrelevant), 1 (partially relevant), and 2 (directly relevant).

In [ ]:
client = AsyncOpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

In [ ]:
async def deepseek_model_chunks(system_prompt, user_prompt):
    """Generates Question-Thought-Answer triplets from the given text using OpenAI."""
    try:
        response = await client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            # response_format={"type": "json_object"}
        )

        # print(response)

        # Attempt to parse the response as JSON
        # try:
        #     rankings = json.loads(response.choices[0].message.content)
        #     return rankings
        # except json.JSONDecodeError as e:
        #     print(f"Error decoding JSON from model response: {e}")
        #     print("Raw response text:", response.choices[0].message.content)
        #     return []

        return response

    except Exception as e:
        print(f"An error occurred during ranking generation: {e}")
        return []

In [ ]:
deepseek_prompts = []

system_prompt_chunk = """You are a helpful assistant that provides financial document chunk rankings based on which chunk is most likely to contain the information relevant to the question. Do not add any thinking, reasoning or other texts."""

for index, row in df_chunk_eval.iterrows():
    eval_question_chunk = row['chunks']

    prompt = f"""Evaluate the relevance ranking of the chunks for the following question based on how relevant the content in chunk is for answering the question.\n\n{eval_question_chunk}\n\n"""

    prompt += """Your Task:\nRank the chunks on one of the three numbers: 0 (chunk is irrelevant in answering the question asked), 1 (chunk is partially relevant in answering the question asked), and 2 (chunk is directly relevant in answering the question asked). Return the result rankings in a JSON format in the form: {'0':1, '1':0..} where keys are the chunk numbers and values are the ranks assigned.\nAnswer Ranking:"""

    deepseek_prompts.append(prompt)

df_chunk_eval['deepseek_prompts'] = deepseek_prompts

print("DeepSeek prompts generated and added to df_doc_eval.")
display(df_chunk_eval.head())

DeepSeek prompts generated and added to df_doc_eval.


,_id,messages,chunks,deepseek_prompts
0,chunk_q613509,"[{'role': 'user', 'content': 'Identify the 10 ...",Question: What guidance was offered on Ball Co...,Evaluate the relevance ranking of the chunks f...
1,chunk_q83d560,"[{'role': 'user', 'content': 'Identify the 10 ...",Question: How has the ratio of BlackRock’s rec...,Evaluate the relevance ranking of the chunks f...
2,chunk_qc06926,"[{'role': 'user', 'content': 'Identify the 10 ...",Question: How has Caterpillar Inc.’s construct...,Evaluate the relevance ranking of the chunks f...
3,chunk_qaae0f2,"[{'role': 'user', 'content': 'Identify the 10 ...",Question: How do global economic or geopolitic...,Evaluate the relevance ranking of the chunks f...
4,chunk_q059068,"[{'role': 'user', 'content': 'Identify the 10 ...",Question: What did Chubb Limited’s leadership ...,Evaluate the relevance ranking of the chunks f...


In [ ]:
df_chunk_eval["deepseek_prompts"][0]

"Evaluate the relevance ranking of the chunks for the following question based on how relevant the content in chunk is for answering the question.\n\nQuestion: What guidance was offered on Ball Corporation’s aluminum can production capacity targets?\nText chunks:\n[Chunk Index 0] PART I. FINANCIAL INFORMATION\n\nItem 1. FINANCIAL STATEMENTS\n\nBALL CORPORATION\n\nUNAUDITED CONDENSED CONSOLIDATED STATEMENTS OF EARNINGS\n[Chunk Index 1] ######Three Months Ended September 30,##########Nine Months Ended September 30,####\n($ in millions, except per share amounts)####2023######2022####2023######2022\nNet sales##$##3,571####$##3,951##$##10,626####$##11,801\nCosts and expenses####################\nCost of sales (excluding depreciation and amortization)####(2,894)######(3,275)####(8,655)######(9,736)\nDepreciation and amortization####(173)######(157)####(509)######(510)\nSelling, general and administrative####(132)######(159)####(428)######(506)\nBusiness consolidation and other activities####

In [ ]:
df_chunk_eval["messages"][0][0]['content']

"Identify the 10 most relevant text chunks for answering this question, then rank them in order of relevance (best first).\nQuestion: What guidance was offered on Ball Corporation’s aluminum can production capacity targets?\nText chunks:\n[Chunk Index 0] PART I. FINANCIAL INFORMATION\n\nItem 1. FINANCIAL STATEMENTS\n\nBALL CORPORATION\n\nUNAUDITED CONDENSED CONSOLIDATED STATEMENTS OF EARNINGS\n[Chunk Index 1] ######Three Months Ended September 30,##########Nine Months Ended September 30,####\n($ in millions, except per share amounts)####2023######2022####2023######2022\nNet sales##$##3,571####$##3,951##$##10,626####$##11,801\nCosts and expenses####################\nCost of sales (excluding depreciation and amortization)####(2,894)######(3,275)####(8,655)######(9,736)\nDepreciation and amortization####(173)######(157)####(509)######(510)\nSelling, general and administrative####(132)######(159)####(428)######(506)\nBusiness consolidation and other activities####(47)######163####(61)#####

In [ ]:
"""calculate max len of strings in df_chunk_eval["messages"][x][0]['content'] where x is row"""
max_len = 0
for x in range(len(df_chunk_eval)):
  if len(df_chunk_eval["messages"][x][0]['content']) > max_len:
    max_len = len(df_chunk_eval["messages"][x][0]['content'])
max_len

1050493

In [ ]:
i = df_chunk_eval["messages"][0][0]['content']
ranking_chunk = deepseek_model_chunks(system_prompt_chunk, i)
print(ranking_chunk)

<coroutine object deepseek_model_chunks at 0x7d0b11896d40>


In [ ]:
# convert string of a list to list
import ast
ast.literal_eval(ranking_chunk.choices[0].message.content)

[92, 93, 94, 95, 96, 97, 98, 99, 100, 101]

In [ ]:
print(ranking_chunk.choices[0].message.content)

[92, 93, 94, 95, 96, 97, 98, 99, 100, 101]


## Get Chunks ranking

In [ ]:
def create_chunk_prompt_top_k(question: str, chunks: List[str], chunk_indices: List[int], k: int = 10) -> str:
    """Ask model to select and rank only top-k most relevant chunks"""
    # Use k if chunks length > 10, else use chunks length
    actual_k = k if len(chunks) > 10 else len(chunks)

    prompt = f"""Identify the {actual_k} most relevant text chunks for answering this question, then rank them in order of relevance (best first).
Question: {question}
Text chunks:
"""
    for i, (chunk, orig_idx) in enumerate(zip(chunks, chunk_indices)):
        prompt += f"[Chunk Index {orig_idx}] {chunk}\n"
    prompt += f"""
Task: Select and rank the {actual_k} most relevant chunks among the given text chunks.
- Put the BEST chunk first
- Put the 2nd best chunk second
- Continue until you have ranked your top {actual_k} chunks
Response Format: [1st_most_relevant_index, 2nd_most_relevant_index, ..., {actual_k}th_most_relevant_index]"""

    return prompt

In [ ]:
async def process_chunk_ranking_two_stage(system_prompt, messages):
  """Process chunk ranking with multi-stage approach for high token count cases"""
  try:
      # Check if this is a high token case by examining the message content
      encoding = tiktoken.get_encoding("cl100k_base")
      # content = messages[0].get('content', '')
      content = messages
      token_count = len(encoding.encode(content))

      if token_count > 120000:

          # Extract question and chunks from the message content
          # Find question
          question_start = content.find('Question:')
          question_end = content.find('\n', question_start)
          if question_start != -1 and question_end != -1:
              question = content[question_start + len('Question:'):question_end].strip()
          else:
              question = None

          # Find chunks using regex-like pattern matching
          chunks = []
          chunk_indices = []

          import re
          # Pattern to match [Chunk Index N] followed by content until next [Chunk Index] or Task:
          chunk_pattern = r'\[Chunk Index (\d+)\]\s*([\s\S]*?)(?=\[Chunk Index|Task:|$)'
          matches = re.findall(chunk_pattern, content)

          for i, match in enumerate(matches):
              orig_idx = int(match[0])
              chunk_content = match[1].strip()

              # Clean up chunk content - remove any task instructions that might be caught
              if 'Task:' in chunk_content:
                  chunk_content = chunk_content.split('Task:')[0].strip()

              if chunk_content:
                  chunks.append(chunk_content)
                  chunk_indices.append(orig_idx)

          if not question or not chunks:
              print("⚠️ Could not parse question and chunks, falling back to normal processing")
              final_response = await deepseek_model_chunks(system_prompt, messages)
              # response = await get_model_response(messages, semaphore=semaphore)
              # predicted_ranking = extract_ranking_from_response(response, 10)
          else:
              # Split chunks into three parts
              third_point_1 = len(chunks) // 3
              third_point_2 = (len(chunks) * 2) // 3

              # First third
              first_third_chunks = chunks[:third_point_1]
              first_third_indices = chunk_indices[:third_point_1]
              first_prompt = create_chunk_prompt_top_k(question, first_third_chunks, first_third_indices, k=10)
              # first_messages = [{"role": "user", "content": first_prompt}]
              # first_response = await get_model_response(first_messages, semaphore=semaphore)
              first_top_3 = await deepseek_model_chunks(system_prompt, first_prompt)
              first_top_3 = ast.literal_eval(first_top_3.choices[0].message.content)
              # first_top_3 = extract_ranking_from_response(first_response, 10)

              # Second third
              second_third_chunks = chunks[third_point_1:third_point_2]
              second_third_indices = chunk_indices[third_point_1:third_point_2]
              second_prompt = create_chunk_prompt_top_k(question, second_third_chunks, second_third_indices, k=10)
              # second_messages = [{"role": "user", "content": second_prompt}]
              # second_response = await get_model_response(second_messages, semaphore=semaphore)
              second_top_3 = await deepseek_model_chunks(system_prompt, second_prompt)
              second_top_3 = ast.literal_eval(second_top_3.choices[0].message.content)
              # second_top_3 = extract_ranking_from_response(second_response, 10)

              # Third third
              third_third_chunks = chunks[third_point_2:]
              third_third_indices = chunk_indices[third_point_2:]
              third_prompt = create_chunk_prompt_top_k(question, third_third_chunks, third_third_indices, k=10)
              # third_messages = [{"role": "user", "content": third_prompt}]
              # third_response = await get_model_response(third_messages, semaphore=semaphore)
              third_top_4 = await deepseek_model_chunks(system_prompt, third_prompt)
              third_top_4 = ast.literal_eval(third_top_4.choices[0].message.content)
              # third_top_4 = extract_ranking_from_response(third_response, 10)

              # Combine top results from each third
              combined_indices = first_top_3 + second_top_3 + third_top_4
              combined_chunks = []

              # Get chunks for the combined indices while preserving original indices
              for idx in combined_indices:
                  if idx in chunk_indices:
                      chunk_pos = chunk_indices.index(idx)
                      combined_chunks.append(chunks[chunk_pos])

              final_prompt = create_chunk_prompt_top_k(question, combined_chunks, combined_indices, k=10)
              # final_messages = [{"role": "user", "content": final_prompt}]
              final_response = await deepseek_model_chunks(system_prompt, final_prompt) #, semaphore=semaphore)
              # predicted_ranking = extract_ranking_from_response(final_response, 10)

      else:
          # Normal single-stage processing
          final_response = await deepseek_model_chunks(system_prompt, messages) #, semaphore=semaphore)
          # predicted_ranking = extract_ranking_from_response(response, 10)

      return final_response
  except Exception as e:
      traceback.print_exc()
      print(f"❌ Error processing chunk ranking item: {e}")
      return []

In [ ]:
# results = pd.DataFrame(columns=['sample_id', 'target_index'])

In [ ]:
rankings_chunk = []
for i in range(115, len(df_chunk_eval)):
  prompt = df_chunk_eval["messages"][i][0]['content']
  ranking_chunk = await process_chunk_ranking_two_stage(system_prompt_chunk, prompt)
  current_rank_chunks = ast.literal_eval(ranking_chunk.choices[0].message.content)
  rankings_chunk.append(current_rank_chunks)

  if len(current_rank_chunks)<=5:
    current_rank_chunks = current_rank_chunks + [0]*(5-len(current_rank_chunks))

  print(current_rank_chunks)
  for j in current_rank_chunks[:5]:
    current_id = df_chunk_eval['_id'][i]
    chunk_rank = {'sample_id': current_id, 'target_index': j}
    # add chunk rank to df
    results = pd.concat([results, pd.DataFrame([chunk_rank])], ignore_index=True)

df_chunk_eval["rankings"] = rankings_chunk

[32, 33, 34, 35, 38, 39, 40, 89, 90, 114]
[7, 6, 9, 1, 16, 46, 47, 44, 45, 240]
[10, 2, 0, 9, 4, 11, 6, 8, 3, 7]
[5, 4, 6, 7, 24, 25, 26, 27, 28, 29]
[1, 0, 2, 27, 28, 29, 30, 31, 32, 33]
[4, 7, 9, 10, 11, 28, 30, 1, 8, 12]
[5, 7, 57, 67, 77, 91, 101, 107, 19, 35]
[67, 65, 66, 68, 69, 70, 71, 72, 64, 73]
[2, 3, 9, 8, 30, 31, 32, 33, 34, 35]
[75, 15, 142, 137, 138, 139, 140, 141, 0, 1]
[14, 15, 27, 33, 57, 59, 5, 7, 51, 13]
[194, 195, 196, 197, 198, 199, 115, 116, 117, 118]
[1, 107, 108, 109, 110, 111, 112, 113, 114, 115]
[60, 61, 128, 129, 7, 62, 63, 64, 65, 66]
[52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
[101, 126, 55, 9, 144, 145, 146, 138, 139, 142]
[38, 42, 487, 486, 485, 39, 40, 43, 44, 41]
[1, 6, 4, 10, 5, 0, 2, 3, 7, 9]
[6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[5, 7, 3, 4, 6, 122, 123, 124, 125, 0]
[15, 16, 17, 18, 1, 2, 4, 5, 6, 7]
[97, 96, 110, 111, 112, 101, 108, 185, 186, 98]
[43, 42, 41, 44, 45, 47, 48, 49, 50, 51]
[6, 7, 4, 1, 2, 3, 5, 0, 8, 9]
[0, 5, 15, 1, 37, 38, 45, 46, 48, 4

ValueError: Length of values (85) does not match length of index (200)

In [ ]:
ranking_list_temp = """
[96, 98, 99, 100, 102, 103, 104, 106, 107, 108]
[8, 9, 137, 138, 139, 140, 141, 142, 143, 144]
[7, 5, 12, 45, 33, 19, 61, 67, 13, 15]
[2, 3, 4, 5, 6, 7, 8, 181, 182, 1]
[21, 22, 20, 6, 5, 1, 2, 3, 4, 7]
[5, 7, 57, 113, 57, 5, 7, 113, 5, 7]
[11, 17, 23, 29, 35, 43, 49, 55, 61, 67]
[4, 1, 3, 5, 9, 6, 2, 0, 7, 8]
[150, 149, 41, 230, 233, 231, 232, 234, 235, 236]
[29, 30, 31, 32, 33, 34, 35, 36, 37, 38]
[77, 76, 90, 89, 91, 56, 57, 78, 79, 80]
[3, 2, 1, 4, 5, 6, 7, 8, 9, 10]
[9, 66, 68, 16, 6, 8, 5, 2, 1, 15]
[4, 20, 19, 24, 25, 3, 0, 1, 2, 5]
[31, 30, 29, 28, 32, 33, 34, 35, 36, 37]
[43, 44, 45, 46, 47, 48, 49, 50, 51, 52]
[7, 5, 89, 87, 91, 93, 101, 103, 105, 107]
[5, 7, 11, 57, 23, 27, 61, 67, 89, 101]
[1, 3, 11, 9, 6, 12, 7, 4, 13, 2]
[3, 1, 0, 2, 4]
[17, 6, 5, 16, 15, 18, 19, 20, 21, 22]
[10, 13, 16, 9, 12, 15, 14, 11, 18, 17]
[253, 647, 4, 42, 64, 75, 237, 238, 252, 489]
[3, 0, 23, 24, 25, 26, 27, 4, 1, 2]
[16, 14, 13, 15, 11, 12, 1, 2, 10, 9]
[86, 29, 30, 95, 96, 26, 92, 87, 3, 4]
[2, 3, 4, 5, 6, 7, 8, 9, 10, 0]
[34, 36, 37, 38, 39, 40, 41, 42, 46, 47]
[99, 101, 103, 45, 47, 49, 79, 81, 23, 27]
[1, 3, 0, 2, 12, 13, 7, 8, 9, 10]
[37, 35, 36, 40, 39, 0, 1, 2, 3, 4]
[13, 17, 23, 33, 43, 55, 65, 75, 85, 111]
[29, 30, 31, 32, 33, 34, 35, 36, 37, 38]
[41, 50, 51, 56, 57, 58, 59, 60, 61, 64]
[11, 4, 3, 1, 45, 40, 39, 38, 37, 36]
[7, 5, 3, 11, 13, 15, 17, 19, 21, 23]
[13, 238, 244, 245, 242, 243, 241, 240, 239, 237]
[2, 8, 9, 10, 1, 0, 3, 4, 7, 6]
[19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
[1, 2, 11, 12, 46, 47, 48, 54, 55, 56]
[45, 46, 47, 48, 49, 50, 51, 52, 53, 54]
[17, 116, 119, 120, 121, 122, 123, 124, 125, 126]
[0, 1, 2, 3, 4]
[7, 20, 21, 11, 12, 15, 16, 17, 18, 19]
[87, 88, 89, 116, 117, 118, 119, 120, 121, 122]
[39, 40, 41, 42, 43, 44, 45, 46, 47, 48]
[44, 32, 45, 46, 47, 48, 49, 50, 63, 70]
[33, 59, 60, 62, 44, 42, 41, 39, 40, 37]
[47, 48, 46, 49, 50, 51, 52, 53, 54, 55]
[57, 56, 131, 130, 128, 132, 87, 127, 86, 83]
[3, 0, 2, 1, 12, 18, 34, 37, 38, 6]
[69, 33, 34, 36, 37, 38, 39, 40, 41, 42]
[22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
[5, 7, 3, 13, 15, 23, 29, 39, 41, 45]
[5, 7, 23, 41, 65, 47, 51, 35, 39, 61]
[1, 3, 11, 4, 9, 6, 7, 8, 12, 13]
[7, 5, 17, 23, 31, 45, 59, 81, 93, 71]
[5, 85, 21, 73, 51, 53, 55, 61, 67, 81]
[46, 45, 47, 59, 60, 61, 56, 57, 58, 44]
[13, 15, 18, 19, 20, 21, 22, 23, 26, 27]
[38, 39, 40, 53, 54, 55, 56, 57, 58, 59]
[12, 13, 50, 51, 90, 94, 95, 9, 10, 5]
[88, 89, 90, 91, 92, 93, 94, 95, 96, 97]
[34, 61, 52, 58, 75, 86, 60, 64, 70, 72]
[30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
[49, 50, 51, 52, 53, 54, 55, 56, 57, 58]
[45, 46, 44, 48, 49, 50, 51, 52, 53, 47]
[27, 349, 350, 381, 382, 26, 197, 18, 56, 57]
[25, 5, 7, 9, 31, 33, 39, 45, 63, 65]
[20, 21, 17, 18, 19, 1, 8, 9, 10, 11]
[11, 12, 10, 9, 8, 7, 6, 5, 4, 3]
[12, 25, 26, 28, 1, 3, 8, 10, 21, 23]
[50, 12, 11, 10, 9, 8, 7, 6, 5, 4]
[15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
[13, 3, 5, 9, 11, 7, 15, 17, 19, 21]
[7, 5, 9, 13, 15, 19, 21, 23, 25, 27]
[89, 90, 91, 92, 93, 94, 95, 96, 97, 98]
[55, 8, 60, 61, 69, 68, 67, 66, 65, 64]
[90, 91, 92, 94, 95, 96, 97, 98, 99, 100]
[73, 126, 142, 125, 92, 94, 95, 110, 111, 112]
[85, 27, 25, 87, 26, 43, 45, 5, 29, 33]
[66, 67, 65, 64, 10, 11, 9, 8, 7, 6]
[49, 48, 47, 46, 45, 44, 43, 42, 41, 40]
[16, 18, 17, 19, 15, 14, 22, 20, 21, 23]
[137, 138, 139, 140, 141, 142, 143, 144, 145, 146]
[3, 4, 33, 35, 36, 37, 38, 39, 40, 41]
[9, 10, 7, 6, 0, 2, 1, 3, 4, 5]
[1, 0, 7, 6, 8, 9, 10, 11, 12, 13]
[3, 9, 12, 13, 23, 49, 53, 59, 63, 2]
[0, 1, 33, 34, 59, 61, 62, 64, 65, 66]
[26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
[6, 5, 4, 7, 8, 3, 2, 1, 0, 9]
[5, 7, 45, 51, 67, 69, 17, 19, 33, 57]
[12, 16, 108, 55, 56, 57, 58, 59, 60, 61]
[69, 68, 27, 70, 29, 4, 0, 1, 2, 3]
[14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
[37, 38, 39, 42, 43, 36, 34, 45, 46, 48]
[7, 5, 4, 6, 3, 8, 9, 10, 11, 12]
[58, 59, 61, 62, 57, 56, 60, 52, 19, 64]
[48, 49, 42, 43, 44, 45, 46, 47, 73, 50]
[5, 7, 3, 9, 19, 21, 23, 25, 33, 37]
[17, 19, 16, 18, 6, 7, 63, 64, 65, 4]
[41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
[9, 7, 6, 8, 0, 2, 5, 17, 20, 4]
[23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
[144, 145, 143, 142, 141, 140, 139, 138, 137, 136]
[5, 63, 62, 65, 43, 42, 64, 77, 14, 15]
[31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
[47, 48, 49, 50, 51, 52, 53, 54, 55, 56]
[3, 8, 2, 9, 18, 11, 15, 19, 20, 23]
[13, 12, 14, 58, 59, 60, 61, 62, 63, 64]
[181, 120, 117, 114, 115, 116, 118, 119, 121, 122]
[23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
[0, 1, 2, 3, 4]
[32, 33, 34, 35, 38, 39, 40, 89, 90, 114]
[7, 6, 9, 1, 16, 46, 47, 44, 45, 240]
[10, 2, 0, 9, 4, 11, 6, 8, 3, 7]
[5, 4, 6, 7, 24, 25, 26, 27, 28, 29]
[1, 0, 2, 27, 28, 29, 30, 31, 32, 33]
[4, 7, 9, 10, 11, 28, 30, 1, 8, 12]
[5, 7, 57, 67, 77, 91, 101, 107, 19, 35]
[67, 65, 66, 68, 69, 70, 71, 72, 64, 73]
[2, 3, 9, 8, 30, 31, 32, 33, 34, 35]
[75, 15, 142, 137, 138, 139, 140, 141, 0, 1]
[14, 15, 27, 33, 57, 59, 5, 7, 51, 13]
[194, 195, 196, 197, 198, 199, 115, 116, 117, 118]
[1, 107, 108, 109, 110, 111, 112, 113, 114, 115]
[60, 61, 128, 129, 7, 62, 63, 64, 65, 66]
[52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
[101, 126, 55, 9, 144, 145, 146, 138, 139, 142]
[38, 42, 487, 486, 485, 39, 40, 43, 44, 41]
[1, 6, 4, 10, 5, 0, 2, 3, 7, 9]
[6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[5, 7, 3, 4, 6, 122, 123, 124, 125, 0]
[15, 16, 17, 18, 1, 2, 4, 5, 6, 7]
[97, 96, 110, 111, 112, 101, 108, 185, 186, 98]
[43, 42, 41, 44, 45, 47, 48, 49, 50, 51]
[6, 7, 4, 1, 2, 3, 5, 0, 8, 9]
[0, 5, 15, 1, 37, 38, 45, 46, 48, 49]
[83, 89, 41, 23, 103, 57, 55, 99, 45, 93]
[5, 7, 17, 23, 25, 33, 37, 45, 47, 63]
[11, 12, 0, 3, 8, 16, 35, 37, 151, 153]
[194, 112, 117, 193, 113, 114, 119, 135, 134, 130]
[10, 175, 178, 180, 183, 186, 200, 205, 210, 213]
[3, 2, 1, 0, 4, 12, 16, 33, 32, 34]
[30, 35, 21, 26, 22, 25, 29, 33, 31, 23]
[2, 3, 6, 7, 15, 16, 17, 18, 19, 20]
[59, 60, 61, 62, 63, 64, 65, 66, 67, 68]
[64, 65, 66, 73, 74, 75, 76, 77, 78, 79]
[1, 5, 8, 21, 0, 3, 2, 4, 6, 7]
[9, 27, 26, 28, 29, 30, 31, 32, 33, 34]
[7, 5, 21, 15, 33, 45, 57, 63, 69, 91]
[111, 112, 113, 114, 115, 116, 117, 160, 161, 162]
[9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
[125, 88, 87, 12, 11, 65, 66, 7, 235, 238]
[19, 25, 27, 40, 41, 64, 65, 72, 73, 102]
[37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
[3, 11, 13, 45, 47, 95, 101, 125, 5, 57]
[5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[3, 4, 1, 6, 7, 8, 9, 10, 11, 0]
[20, 22, 19, 16, 13, 11, 10, 9, 8, 122]
[3, 4, 0, 2, 1]
[9, 8, 101, 108, 107, 106, 105, 104, 100, 103]
[15, 20, 19, 16, 14, 17, 18, 13, 12, 21]
[63, 68, 64, 65, 66, 67, 69, 72, 73, 75]
[13, 30, 18, 14, 15, 17, 19, 20, 21, 23]
[32, 18, 38, 8, 33, 34, 35, 36, 37, 39]
[25, 5, 10, 3, 4, 24, 171, 138, 7, 8]
[31, 32, 34, 35, 36, 37, 38, 39, 40, 41]
[30, 95, 128, 129, 96, 125, 126, 127, 1, 5]
[5, 15, 21, 29, 79, 85, 101, 111, 3, 7]
[16, 14, 2, 1, 6, 5, 3, 17, 15, 0]
[12, 11, 10, 13, 14, 15, 16, 17, 18, 19]
[2, 12, 11, 30, 31, 32, 33, 34, 35, 36]
[21, 24, 23, 22, 25, 20, 26, 27, 18, 19]
[79, 80, 81, 85, 89, 90, 91, 92, 93, 94]
[16, 17, 18, 19, 20, 21, 22, 23, 12, 13]
[5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
[64, 65, 66, 67, 68, 69, 70, 71, 72, 73]
[45, 44, 43, 42, 41, 40, 39, 38, 37, 36]
[3, 5, 13, 27, 37, 39, 49, 59, 71, 73]
[366, 367, 354, 355, 368, 370, 371, 372, 373, 39]
[40, 35, 36, 37, 38, 39, 41, 42, 43, 44]
[34, 33, 32, 35, 36, 37, 38, 39, 40, 41]
[1, 2, 0, 3, 4, 5, 6, 7, 8, 9]
[10, 2, 8, 9, 11, 6, 15, 20, 21, 25]
[9, 8, 4, 7, 2, 1, 0, 3, 5, 6]
[49, 50, 51, 90, 91, 89, 12, 13, 10, 94]
[112, 113, 114, 115, 116, 117, 118, 119, 120, 121]
[26, 27, 28, 56, 57, 58, 59, 60, 61, 72]
[14, 60, 6, 18, 19, 63, 64, 65, 67, 77]
[88, 89, 16, 91, 90, 17, 19, 86, 87, 123]
[28, 27, 29, 40, 41, 42, 43, 44, 45, 46]
[10, 9, 55, 40, 39, 11, 7, 33, 34, 35]
[4, 6, 17, 5, 7, 2, 0, 19, 11, 1]
[62, 63, 66, 67, 94, 95, 96, 97, 98, 100]
[137, 5, 11, 47, 97, 27, 61, 77, 103, 119]
[7, 3, 5, 9, 19, 45, 49, 69, 79, 89]
[17, 0, 23, 25, 16, 9, 8, 1, 3, 4]
"""

In [ ]:
# Convert string to list of lists
ranking_lists_all = [ast.literal_eval(line) for line in ranking_list_temp.strip().splitlines() if line.strip()]

print(ranking_lists_all)

[[96, 98, 99, 100, 102, 103, 104, 106, 107, 108], [8, 9, 137, 138, 139, 140, 141, 142, 143, 144], [7, 5, 12, 45, 33, 19, 61, 67, 13, 15], [2, 3, 4, 5, 6, 7, 8, 181, 182, 1], [21, 22, 20, 6, 5, 1, 2, 3, 4, 7], [5, 7, 57, 113, 57, 5, 7, 113, 5, 7], [11, 17, 23, 29, 35, 43, 49, 55, 61, 67], [4, 1, 3, 5, 9, 6, 2, 0, 7, 8], [150, 149, 41, 230, 233, 231, 232, 234, 235, 236], [29, 30, 31, 32, 33, 34, 35, 36, 37, 38], [77, 76, 90, 89, 91, 56, 57, 78, 79, 80], [3, 2, 1, 4, 5, 6, 7, 8, 9, 10], [9, 66, 68, 16, 6, 8, 5, 2, 1, 15], [4, 20, 19, 24, 25, 3, 0, 1, 2, 5], [31, 30, 29, 28, 32, 33, 34, 35, 36, 37], [43, 44, 45, 46, 47, 48, 49, 50, 51, 52], [7, 5, 89, 87, 91, 93, 101, 103, 105, 107], [5, 7, 11, 57, 23, 27, 61, 67, 89, 101], [1, 3, 11, 9, 6, 12, 7, 4, 13, 2], [3, 1, 0, 2, 4], [17, 6, 5, 16, 15, 18, 19, 20, 21, 22], [10, 13, 16, 9, 12, 15, 14, 11, 18, 17], [253, 647, 4, 42, 64, 75, 237, 238, 252, 489], [3, 0, 23, 24, 25, 26, 27, 4, 1, 2], [16, 14, 13, 15, 11, 12, 1, 2, 10, 9], [86, 29, 30, 9

In [ ]:
len(ranking_lists_all)

200

In [ ]:
results = pd.DataFrame(columns=['sample_id', 'target_index'])


for index, i in enumerate(ranking_lists_all):
  if len(i)<=5:
      i = i + [0]*(5-len(i))

  # print(i)
  for j in i[:5]:
    current_id = df_chunk_eval['_id'][index]
    chunk_rank = {'sample_id': current_id, 'target_index': j}
    # add chunk rank to df
    results = pd.concat([results, pd.DataFrame([chunk_rank])], ignore_index=True)

In [ ]:
results

,sample_id,target_index
0,chunk_q613509,96
1,chunk_q613509,98
2,chunk_q613509,99
3,chunk_q613509,100
4,chunk_q613509,102
...,...,...
995,chunk_qa1d2bd,17
996,chunk_qa1d2bd,0
997,chunk_qa1d2bd,23
998,chunk_qa1d2bd,25


In [ ]:
# save results with chunks
results.to_csv('results_chunks.csv', index=False)

In [ ]:
# join results.csv and results_chunks.csv
df_results = pd.read_csv('results.csv')

# change ranking col to target_index
df_results = df_results.rename(columns={'rankings': 'target_index'}, inplace=False)
df_results_chunks = pd.read_csv('results_chunks.csv')

# join
df_results_all = pd.concat([df_results, df_results_chunks], ignore_index=True)

In [ ]:
df_results_all

,sample_id,target_index
0,doc_q39d7b7,0
1,doc_q39d7b7,3
2,doc_q39d7b7,4
3,doc_q39d7b7,2
4,doc_q39d7b7,1
...,...,...
1995,chunk_qa1d2bd,17
1996,chunk_qa1d2bd,0
1997,chunk_qa1d2bd,23
1998,chunk_qa1d2bd,25


In [ ]:
df_results_all.to_csv('results_all.csv', index=False)

# Config

# Search for “Databricks free trial” and sign up.

1. Use Model Serving to set up a Databricks ***model endpoint***,
2. Prepare an access token from the ***Settings page***.

In [ ]:
## Configuration and Model Setup

from google.colab import userdata

# Environment variables
DATABRICKS_TOKEN = userdata.get('DATABRICKS_TOKEN') # os.environ.get('DATABRICKS_TOKEN')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') # os.environ.get('OPENAI_API_KEY')

# Check if environment variables are set
if not DATABRICKS_TOKEN:
    print("⚠️ DATABRICKS_TOKEN not found in environment variables")
    print("Please set your Databricks token in your environment or .env file")
if not OPENAI_API_KEY:
    print("⚠️ OPENAI_API_KEY not found in environment variables")
    print("Please set your OpenAI API key in your environment or .env file")

# Initialize clients
client = AsyncOpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://dbc-2af8acef-5331.cloud.databricks.com/serving-endpoints"
)

openai_client = AsyncOpenAI(
    api_key=OPENAI_API_KEY,
)

print("✅ Clients initialized successfully")

✅ Clients initialized successfully


# Pydantic Models for Structured Output

In [ ]:
class Format(BaseModel):
    answer: List[int]

# Functions

Loads a JSONL file and returns a list of dictionaries.

- Opens the file and parses each line as JSON.  
- Handles missing files or parsing errors gracefully (prints error and returns `[]`).  
- Prints the number of items successfully loaded.

Useful for quickly reading evaluation datasets stored in JSONL format.

In [ ]:
def load_evaluation_data(filepath: str) -> List[Dict]:
    """Load evaluation data from JSONL file"""
    data = []
    total_count = 0
    print(f"📁 Loading data from: {filepath}")

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                total_count += 1
                item = json.loads(line.strip())
                data.append(item)
    except FileNotFoundError:
        print(f"❌ File not found: {filepath}")
        return []
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return []

    print(f"✅ Total items loaded: {len(data)}")
    return data

### Function: `create_chunk_prompt_top_k`

This function builds a **prompt string** that asks a language model to pick and rank the most relevant text chunks for a given question.

**Key Steps**
1. **Determine Top-k Size**  
   - Uses `k` (default 10) if there are more than 10 chunks.  
   - Otherwise uses the total number of chunks.

2. **Compose Prompt**  
   - Starts with instructions to select and rank the `actual_k` most relevant chunks.  
   - Inserts the **question** and all provided text chunks, each labeled with its original index.

3. **Specify Output Format**  
   - Requests a final ordered list of chunk indices, e.g.  
     `[1st_most_relevant_index, 2nd_most_relevant_index, ..., kth_most_relevant_index]`.

**Purpose**  
This prompt can be sent to a large language model to **identify and rank the most relevant pieces of text** (chunks) before downstream tasks like question answering or summarization.

In [ ]:
def create_chunk_prompt_top_k(question: str, chunks: List[str], chunk_indices: List[int], k: int = 10) -> str:
    """Ask model to select and rank only top-k most relevant chunks"""
    # Use k if chunks length > 10, else use chunks length
    actual_k = k if len(chunks) > 10 else len(chunks)

    prompt = f"""Identify the {actual_k} most relevant text chunks for answering this question, then rank them in order of relevance (best first).
Question: {question}
Text chunks:
"""
    for i, (chunk, orig_idx) in enumerate(zip(chunks, chunk_indices)):
        prompt += f"[Chunk Index {orig_idx}] {chunk}\n"
    prompt += f"""
Task: Select and rank the {actual_k} most relevant chunks among the given text chunks.
- Put the BEST chunk first
- Put the 2nd best chunk second
- Continue until you have ranked your top {actual_k} chunks
Response Format: [1st_most_relevant_index, 2nd_most_relevant_index, ..., {actual_k}th_most_relevant_index]"""

    return prompt

### Function: `get_model_response`

Asynchronously gets a **financial-analysis response** from a specified model and extracts a ranked list.

- Prepends a **system prompt**: *"You are a helpful financial analyst."*  
- Sends combined messages to the target model (default `databricks-gpt-oss-120b`).  
- Extracts and parses the ranking list using an OpenAI helper (`gpt-4o-mini`).  
- Uses an `asyncio.Semaphore` to limit concurrent requests.  
- Returns an empty list on error.

Efficiently retrieves and structures ranking results from the model.

In [ ]:
async def get_model_response(messages: List[Dict], model: str = "databricks-gpt-oss-120b", semaphore: asyncio.Semaphore = None) -> List[int]:
    """Get response from the model with financial analyst system prompt"""
    async with semaphore:
        system_message = {"role": "system", "content": "You are a helpful financial analyst."}

        full_messages = [system_message] + messages

        try:
            response = await client.chat.completions.create(
                messages=full_messages,
                model=model,
            )
            # Get the raw text response
            raw_output = response.choices[0].message.content[1]["text"].strip()

            # Use OpenAI to extract structured ranking
            extraction_response = await openai_client.beta.chat.completions.parse(
                model="gpt-4o-mini",
                messages=[
                    {"role": "user", "content": f"Extract the ranking list from this text. Return only the numbers in order as a list: {raw_output}"}
                ],
                response_format=Format,
            )

            return extraction_response.choices[0].message.parsed.answer

        except Exception as e:
            traceback.print_exc()
            print(f"❌ Error getting model response: {e}")
            # Return default ranking based on number of items expected
            return []

In [ ]:
df_doc_eval.iloc[0]

,0
_id,doc_q39d7b7
messages,"[{'role': 'user', 'content': 'Rank the followi..."
question,How has Salesforce’s subscription and support ...
top_k_dev_ques,[How has PTC Inc.’s software subscription segm...
top_k_dev_qrels,"[{'10-Q': 4, '10-K': 3, '8-K': 2, 'Earnings': ..."


In [ ]:
async def get_model_response_with_examples(messages: List[Dict], model: str = "databricks-gpt-oss-120b"):
    """Get response from the model with financial analyst system prompt"""
    system_message = {"role": "system", "content": "You are a helpful assistant that provides financial document rankings based on which document is most likely to contain the information relevant to the question. Output only the ranking dictionary. Do not add any thinking, reasoning or other texts."}

    eval_question = messages['question']
    top_k_ques = messages['top_k_dev_ques']
    top_k_qrels = messages['top_k_dev_qrels']


    prompt = f"""Evaluate the relevance ranking of financial document types for the following question based on the provided examples.\n\nEvaluation Question: {eval_question}\n\n"""

    for i, qrel_dict in enumerate(top_k_qrels):
        example_question = top_k_ques[i]
        prompt += f"Example {i+1}:\n"
        prompt += f"Question: {example_question}\n"
        prompt += f"Rankings: {qrel_dict}\n\n"

    prompt += """Your Task:\nBased on the examples, rank the documents in the order of which document type is most likely to contain the answer based on the given question. In the given examples, rank 4 is most relevant and rank 0 is least relevant. Return the result ranking in the format {"Doc_name1": rank1, "Doc_name2": rank2,... "Doc_name5": rank5}.\nAnswer Ranking:"""

    user_message = {"role": "user", "content": prompt}

    full_messages = [system_message] + [user_message]

    try:
        response = await client.chat.completions.create(
            messages=full_messages,
            model=model,
        )
        # Get the raw text response
        raw_output = response.choices[0].message.content[1]["text"].strip()

        print(raw_output)
        return raw_output

    except Exception as e:
        traceback.print_exc()
        print(f"❌ Error getting model response: {e}")
        # Return default ranking based on number of items expected
        return []

In [ ]:
doc_rankings_gptoss = []
for i in range(len(df_doc_eval)):
  ranking = await get_model_response_with_examples(df_doc_eval.iloc[i])
  doc_rankings_gptoss.append(ranking)

{"10-Q": 4, "10-K": 3, "8-K": 2, "Earnings": 1, "DEF14A": 0}
{'10-K': 4, '10-Q': 3, '8-K': 2, 'DEF14A': 1, 'Earnings': 0}
{'10-K': 4, 'Earnings': 3, '10-Q': 2, '8-K': 1, 'DEF14A': 0}
{'10-Q': 4, '10-K': 3, 'Earnings': 2, '8-K': 1, 'DEF14A': 0}
{'Earnings': 4, 'DEF14A': 3, '8-K': 2, '10-Q': 1, '10-K': 0}
{'DEF14A': 4, '10-K': 3, 'Earnings': 2, '8-K': 1, '10-Q': 0}
{"10-K": 4, "10-Q": 3, "8-K": 2, "DEF14A": 1, "Earnings": 0}
{"DEF14A": 4, "10-K": 3, "10-Q": 2, "8-K": 1, "Earnings": 0}
{'Earnings': 4, '10-Q': 3, '8-K': 2, 'DEF14A': 1, '10-K': 0}
{"10-K": 4, "DEF14A": 3, "10-Q": 2, "Earnings": 1, "8-K": 0}
{"10-K": 4, "10-Q": 3, "Earnings": 2, "DEF14A": 1, "8-K": 0}
{"Earnings": 3, "10-Q": 2, "8-K": 2, "DEF14A": 1, "10-K": 0}
{'10-K': 4, '10-Q': 3, '8-K': 2, 'Earnings': 1, 'DEF14A': 0}
{'Earnings': 4, '10-K': 3, 'DEF14A': 2, '10-Q': 1, '8-K': 0}
{"DEF14A": 4, "10-K": 3, "10-Q": 2, "Earnings": 1, "8-K": 0}
{"Earnings": 4, "10-K": 3, "10-Q": 2, "DEF14A": 1, "8-K": 0}
{'10-Q': 4, '10-K': 3, '

In [ ]:
# convert back
import ast

df_doc_eval["rankings"] = [ast.literal_eval(i) for i in doc_rankings_gptoss]

# convert ranking back to numbers
name_to_docno = {"DEF14A": "0", "10-K": "1", "10-Q": "2", "8-K": "3", "Earnings": "4"}

for i in range(len(df_doc_eval['rankings'])):
  temp = df_doc_eval['rankings'][i]
  # df_doc_eval['rankings'][i] = [{name_to_docno[k]: v for k, v in temp.items()}]
  df_doc_eval.loc[i, "rankings"] = [{name_to_docno[k]: v for k, v in temp.items()}]

# convert into submission form

#create empty df with columns
results = pd.DataFrame(columns=['sample_id', 'target_index'])

for index, i in enumerate(df_doc_eval['_id']):
  for j in range(4,-1,-1):
    current_id = i
    current_rankings = df_doc_eval['rankings'][index][0]

    # get key of the item in current ranking for which value is j
    for key, value in current_rankings.items():
      if value == j:
        doc_rank = {'sample_id': current_id, 'target_index': key}
        # add doc rank to df
        results = pd.concat([results, pd.DataFrame([doc_rank])], ignore_index=True)

# save results so far to csv
results.to_csv('results_doc_gptoss.csv', index=False)

In [ ]:
# sample 100 questions and their ranks from dev set
df_doc_dev_sample = df_doc_dev.sample(n=100, random_state=42)

# save these as csv
df_doc_dev_sample.to_csv('df_doc_dev_sample.csv', index=False)

### Function: `extract_ranking_from_response`

Ensures the ranking list has the required length.

- If `response` has at least `num_items`, return the first `num_items`.
- Otherwise, pad with default indices (`0,1,2,...`) until the length reaches `num_items`.

Guarantees a fixed-size ranking list for downstream use.

In [ ]:
def extract_ranking_from_response(response: List[int], num_items: int) -> List[int]:
    """Extract ranking list from model response"""
    # Ensure we have the right number of items
    if len(response) >= num_items:
        return response[:num_items]
    else:
        # Pad with default values if needed
        padded = response + list(range(len(response), num_items))
        return padded[:num_items]

### Function: `process_chunk_ranking_two_stage`

Handles **chunk ranking** for prompts with very high token counts using a two-stage strategy.

- **Token Check**  
  - Uses `tiktoken` to count tokens in the first message.  
  - If over 60,000 tokens, switches to multi-stage processing.

- **Multi-Stage Processing (if large)**  
  - Extracts `question` and `[Chunk Index N]` sections via regex.  
  - Splits chunks into three groups (first/second/third).  
  - For each group, calls `get_model_response` to find top candidates.  
  - Merges these top chunks and runs a final `get_model_response` to get the final ranking.

- **Single-Stage Processing (if normal)**  
  - Directly calls `get_model_response` and extracts top-10 ranking.

- **Error Handling**  
  - Logs parsing or runtime errors and returns an empty list on failure.

This approach efficiently **ranks large text sets** by first narrowing candidates in parallel and then re-ranking a smaller subset.

In [ ]:
async def process_chunk_ranking_two_stage(messages: List[Dict], semaphore: asyncio.Semaphore, query_id: str) -> List[int]:
    """Process chunk ranking with multi-stage approach for high token count cases"""
    try:
        # Check if this is a high token case by examining the message content
        encoding = tiktoken.get_encoding("cl100k_base")
        content = messages[0].get('content', '')
        token_count = len(encoding.encode(content))

        if token_count > 60000:

            # Extract question and chunks from the message content
            # Find question
            question_start = content.find('Question:')
            question_end = content.find('\n', question_start)
            if question_start != -1 and question_end != -1:
                question = content[question_start + len('Question:'):question_end].strip()
            else:
                question = None

            # Find chunks using regex-like pattern matching
            chunks = []
            chunk_indices = []

            import re
            # Pattern to match [Chunk Index N] followed by content until next [Chunk Index] or Task:
            chunk_pattern = r'\[Chunk Index (\d+)\]\s*([\s\S]*?)(?=\[Chunk Index|Task:|$)'
            matches = re.findall(chunk_pattern, content)

            for i, match in enumerate(matches):
                orig_idx = int(match[0])
                chunk_content = match[1].strip()

                # Clean up chunk content - remove any task instructions that might be caught
                if 'Task:' in chunk_content:
                    chunk_content = chunk_content.split('Task:')[0].strip()

                if chunk_content:
                    chunks.append(chunk_content)
                    chunk_indices.append(orig_idx)

            if not question or not chunks:
                print("⚠️ Could not parse question and chunks, falling back to normal processing")
                response = await get_model_response(messages, semaphore=semaphore)
                predicted_ranking = extract_ranking_from_response(response, 10)
            else:
                # Split chunks into three parts
                third_point_1 = len(chunks) // 3
                third_point_2 = (len(chunks) * 2) // 3

                # First third
                first_third_chunks = chunks[:third_point_1]
                first_third_indices = chunk_indices[:third_point_1]
                first_prompt = create_chunk_prompt_top_k(question, first_third_chunks, first_third_indices, k=10)
                first_messages = [{"role": "user", "content": first_prompt}]
                first_response = await get_model_response(first_messages, semaphore=semaphore)
                first_top_3 = extract_ranking_from_response(first_response, 10)

                # Second third
                second_third_chunks = chunks[third_point_1:third_point_2]
                second_third_indices = chunk_indices[third_point_1:third_point_2]
                second_prompt = create_chunk_prompt_top_k(question, second_third_chunks, second_third_indices, k=10)
                second_messages = [{"role": "user", "content": second_prompt}]
                second_response = await get_model_response(second_messages, semaphore=semaphore)
                second_top_3 = extract_ranking_from_response(second_response, 10)

                # Third third
                third_third_chunks = chunks[third_point_2:]
                third_third_indices = chunk_indices[third_point_2:]
                third_prompt = create_chunk_prompt_top_k(question, third_third_chunks, third_third_indices, k=10)
                third_messages = [{"role": "user", "content": third_prompt}]
                third_response = await get_model_response(third_messages, semaphore=semaphore)
                third_top_4 = extract_ranking_from_response(third_response, 10)

                # Combine top results from each third
                combined_indices = first_top_3 + second_top_3 + third_top_4
                combined_chunks = []

                # Get chunks for the combined indices while preserving original indices
                for idx in combined_indices:
                    if idx in chunk_indices:
                        chunk_pos = chunk_indices.index(idx)
                        combined_chunks.append(chunks[chunk_pos])

                final_prompt = create_chunk_prompt_top_k(question, combined_chunks, combined_indices, k=10)
                final_messages = [{"role": "user", "content": final_prompt}]
                final_response = await get_model_response(final_messages, semaphore=semaphore)
                predicted_ranking = extract_ranking_from_response(final_response, 10)

        else:
            # Normal single-stage processing
            response = await get_model_response(messages, semaphore=semaphore)
            predicted_ranking = extract_ranking_from_response(response, 10)

        return predicted_ranking
    except Exception as e:
        traceback.print_exc()
        print(f"❌ Error processing chunk ranking item: {e}")
        return []

### Function: `process_single_item`

Processes one evaluation item to obtain a ranked list.

- Calls `get_model_response` with the given messages.  
- Uses `extract_ranking_from_response` to ensure the ranking has `num_items` elements.  
- Returns an empty list if any error occurs.

A simple wrapper for **single-item chunk ranking**.

In [ ]:
async def process_single_item(messages: List[Dict], num_items: int, semaphore: asyncio.Semaphore) -> List[int]:
    """Process a single evaluation item"""
    try:
        # Get model response
        response = await get_model_response(messages, semaphore=semaphore)

        # Extract ranking from response
        predicted_ranking = extract_ranking_from_response(response, num_items)

        return predicted_ranking

    except Exception as e:
        print(f"❌ Error processing item: {e}")
        return []

# EVALUATION

### Functions: `evaluate_chunk_ranking` & `evaluate_document_ranking`

Run **end-to-end evaluation** for chunk and document ranking tasks.

#### `evaluate_chunk_ranking`
- Loads evaluation data and checks availability.
- For each item:
  - Uses `process_chunk_ranking_two_stage` to handle high-token cases.
  - Collects the top 5 ranked chunk indices for submission.
- Displays progress and summary (tasks completed, total submission entries).

#### `evaluate_document_ranking`
- Similar flow for document ranking, but:
  - Uses `process_single_item` (no multi-stage splitting).
  - Also records the top 5 ranked document indices.

Both functions return a **submission-ready list of dictionaries** with  
`sample_id` and `target_index` for each predicted top item.

In [ ]:
async def evaluate_chunk_ranking(data_path: str, semaphore: asyncio.Semaphore) -> List[Dict]:
    """Evaluate chunk ranking task and return submission data"""
    print("\n🔍 CHUNK RANKING EVALUATION")
    print("="*50)
    data = load_evaluation_data(data_path)

    if not data:
        print("❌ No data loaded for chunk ranking")
        return []

    print(f"🎯 Evaluating {len(data)} chunk ranking items...")

    # Create tasks for concurrent processing
    tasks = []
    submission_data = []
    for idx, item in enumerate(data):
        messages = item['messages']
        query_id = item['_id']  # Use original _id from data
        # Use multi-stage chunk ranking process for high token cases
        task = process_chunk_ranking_two_stage(messages, semaphore, query_id)
        tasks.append((task, query_id))

    # Process all tasks with progress bar
    results_list = []
    for task_tuple in tqdm(tasks, desc="🔄 Processing chunk ranking", leave=False):
        task, query_id = task_tuple
        result = await task
        if result:
            results_list.append((result, query_id))
            # Add top 5 results to submission data
            for rank, doc_idx in enumerate(result[:5]):
                submission_data.append({'sample_id': query_id, 'target_index': doc_idx})

    print(f"✅ Completed {len(results_list)} chunk ranking tasks")
    print(f"📊 Generated {len(submission_data)} submission entries")
    return submission_data

In [ ]:
async def evaluate_document_ranking(data_path: str, semaphore: asyncio.Semaphore) -> List[Dict]:
    """Evaluate document ranking task and return submission data"""
    print("\n📄 DOCUMENT RANKING EVALUATION")
    print("="*50)
    data = load_evaluation_data(data_path)

    if not data:
        print("❌ No data loaded for document ranking")
        return []

    print(f"🎯 Evaluating {len(data)} document ranking items...")

    # Create tasks for concurrent processing
    tasks = []
    submission_data = []
    for idx, item in enumerate(data):
        messages = item['messages']
        query_id = item['_id']  # Use original _id from data
        task = process_single_item(messages, 10, semaphore)
        tasks.append((task, query_id))

    # Process all tasks with progress bar
    results_list = []
    for task_tuple in tqdm(tasks, desc="🔄 Processing document ranking", leave=False):
        task, query_id = task_tuple
        result = await task
        if result:
            results_list.append((result, query_id))
            # Add top 5 results to submission data
            for rank, doc_idx in enumerate(result[:5]):
                submission_data.append({'sample_id': query_id, 'target_index': doc_idx})

    print(f"✅ Completed {len(results_list)} document ranking tasks")
    print(f"📊 Generated {len(submission_data)} submission entries")
    return submission_data

### Function: `save_submission_csv`

Saves prediction results to a **CSV file** in the required format.

- Creates a CSV with header: `sample_id, target_index`.  
- Writes each entry from `submission_data` as a row.  
- Prints the output file path and total entry count.  
- Logs an error message if file writing fails.

Convenient for generating **final submission files** from evaluation results.

In [ ]:
def save_submission_csv(submission_data: List[Dict], filename: str):
    """Save submission data to CSV file in the required format"""
    try:
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['sample_id', 'target_index'])

            for entry in submission_data:
                writer.writerow([entry['sample_id'], entry['target_index']])

        print(f"💾 Submission file saved to {filename}")
        print(f"📊 Total entries: {len(submission_data)}")
    except Exception as e:
        print(f"❌ Error saving submission file: {e}")

## Data File Check

In [ ]:
# Check if data files exist
chunk_ranking_path = "./chunk_ranking_kaggle_eval.jsonl"
document_ranking_path = "./document_ranking_kaggle_eval.jsonl"

print("🔍 Checking for required data files...")
print(f"📁 Chunk ranking file: {chunk_ranking_path}")
print(f"   Exists: {'✅' if os.path.exists(chunk_ranking_path) else '❌'}")
print(f"📁 Document ranking file: {document_ranking_path}")
print(f"   Exists: {'✅' if os.path.exists(document_ranking_path) else '❌'}")

if os.path.exists(chunk_ranking_path) and os.path.exists(document_ranking_path):
    print("\n🎉 All required files found! Ready to run evaluation.")
else:
    print("\n⚠️ Missing required data files. Please ensure both files exist in the ./output/ directory.")
    print("   You may need to run the data preparation script first.")

🔍 Checking for required data files...
📁 Chunk ranking file: ./chunk_ranking_kaggle_eval.jsonl
   Exists: ✅
📁 Document ranking file: ./document_ranking_kaggle_eval.jsonl
   Exists: ✅

🎉 All required files found! Ready to run evaluation.


## Main Evaluation Pipeline

In [ ]:
async def main():
    """Main evaluation function"""
    print("\n" + "="*60)
    print("🏆 KAGGLE RANKING EVALUATION PIPELINE")
    print("="*60)
    print("🤖 Model: databricks-gpt-oss-120b")
    print("👨‍💼 System Prompt: You are a helpful financial analyst.")
    print("📄 Output: CSV submission file only")
    print("🔄 Concurrency: 1 simultaneous request")
    print("="*60)

    # Create semaphore for limiting concurrent requests
    semaphore = asyncio.Semaphore(1)

    # Check files exist before starting
    if not os.path.exists(chunk_ranking_path) or not os.path.exists(document_ranking_path):
        print("❌ Required data files not found. Please check the file paths.")
        return

    # Evaluate chunk ranking and document ranking concurrently
    print("\n🚀 Starting evaluation...")
    chunk_task = evaluate_chunk_ranking(chunk_ranking_path, semaphore)
    doc_task = evaluate_document_ranking(document_ranking_path, semaphore)

    # Wait for both evaluations to complete
    print("\n⏳ Running both evaluations concurrently...")
    chunk_submission, doc_submission = await asyncio.gather(chunk_task, doc_task)

    # Combine submission data
    all_submission_data = chunk_submission + doc_submission

    # Save submission CSV
    save_submission_csv(all_submission_data, './kaggle_submission.csv')

    print("\n" + "="*60)
    print("🎊 EVALUATION COMPLETE!")
    print("="*60)
    print(f"🔍 Chunk ranking entries: {len(chunk_submission):,}")
    print(f"📄 Document ranking entries: {len(doc_submission):,}")
    print(f"📊 Total submission entries: {len(all_submission_data):,}")
    print(f"💾 Submission file: kaggle_submission.csv")
    print("\n🚀 Ready for Kaggle submission!")
    print("="*60)

## 🚀 Run the Complete Evaluation

Execute this cell to run the full evaluation pipeline. Make sure you have:

1. ✅ Set up your environment variables (DATABRICKS_TOKEN, OPENAI_API_KEY)
2. ✅ Have the evaluation data files in the `./output/` directory:
   - `chunk_ranking_kaggle_eval.jsonl`
   - `document_ranking_kaggle_eval.jsonl`
3. ✅ Installed all required packages

In [ ]:
# Run the complete evaluation pipeline
await main()


🏆 KAGGLE RANKING EVALUATION PIPELINE
🤖 Model: databricks-gpt-oss-120b
👨‍💼 System Prompt: You are a helpful financial analyst.
📄 Output: CSV submission file only
🔄 Concurrency: 1 simultaneous request

🚀 Starting evaluation...

⏳ Running both evaluations concurrently...

🔍 CHUNK RANKING EVALUATION
📁 Loading data from: ./chunk_ranking_kaggle_eval.jsonl
✅ Total items loaded: 200
🎯 Evaluating 200 chunk ranking items...


🔄 Processing chunk ranking:   0%|          | 0/200 [00:00<?, ?it/s]


📄 DOCUMENT RANKING EVALUATION
📁 Loading data from: ./document_ranking_kaggle_eval.jsonl
✅ Total items loaded: 200
🎯 Evaluating 200 document ranking items...



🔄 Processing chunk ranking:   2%|▏         | 3/200 [00:57<1:05:21, 19.91s/it]
                                                                              
                                                                                 

CancelledError: 

/usr/lib/python3.12/asyncio/base_events.py:2000: RuntimeWarning: coroutine 'process_single_item' was never awaited
  handle = None  # Needed to break cycles when an exception occurs.


## 📊 View Results

Check the generated submission file and its contents:

In [ ]:
import pandas as pd

# Load and display submission file if it exists
submission_file = './kaggle_submission.csv'
if os.path.exists(submission_file):
    df = pd.read_csv(submission_file)
    print(f"📊 Submission file shape: {df.shape}")
    print(f"\n📋 Sample data (first 10 rows):")
    print(df.head(10))
    print(f"\n🎯 Statistics:")
    print(f"   • Unique sample_ids: {df['sample_id'].nunique():,}")
    print(f"   • Sample ID range: {df['sample_id'].min()} to {df['sample_id'].max()}")
    print(f"   • Target index range: {df['target_index'].min()} to {df['target_index'].max()}")
    print(f"   • Total entries: {len(df):,}")

    # Show distribution of entries per sample_id
    entries_per_sample = df.groupby('sample_id').size()
    print(f"\n📈 Entries per sample_id distribution:")
    print(f"   • Mean: {entries_per_sample.mean():.1f}")
    print(f"   • Min: {entries_per_sample.min()}")
    print(f"   • Max: {entries_per_sample.max()}")
    print(f"   • Most common: {entries_per_sample.mode().iloc[0]} entries per sample")

else:
    print("❌ Submission file not found. Please run the evaluation first.")

# Task
Write code to finetune a reranker model on the document ranking dev set.

## Data preparation

### Subtask:
Load the document ranking dev set and prepare the data in a format suitable for training a reranker model.


**Reasoning**:
The subtask requires loading the document ranking development data and transforming it into a format suitable for training. I will load the data, extract the question and document information, and create a list of dictionaries containing the question, document text, and relevance label for each document.



**Reasoning**:
The previous command failed because the 'question' column was not present in the `df_doc_dev` DataFrame. I need to extract the question from the 'messages' column first, as was done in previous successful steps in the notebook. Then I can proceed with preparing the data for reranking.



## Model Training

### Subtask:
Train the chosen reranker model (ColBERT) on the prepared document ranking dev set.

In [ ]:
!pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [ ]:
df_doc_dev

,uuid,messages,qrel,question
0,qe100cdf8e8f5,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}",How has Agilent Technologies’ instrument relia...
1,q723817294fbe,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '0': 2, '1': 1, '3': 1, '2': 0}",How did analysts question the outlook for Agil...
2,qd79970afaa57,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '1': 2, '0': 1, '2': 1, '3': 0}",How do sustainability or ESG considerations in...
3,qeb144d0e9991,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 3, '2': 2, '4': 1, '0': 0, '3': 0}","What exposure does Agilent Technologies, Inc. ..."
4,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '2': 2, '1': 1, '0': 0, '3': 0}",What sentiment trends are visible around Agile...
...,...,...,...,...
4981,q30f7c8056c93,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 4, '2': 3, '0': 2, '4': 1, '3': 0}",Which supply chain trends are affecting availa...
4982,q402c59dfe9b1,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 4, '0': 3, '4': 2, '3': 1, '2': 0}",What themes have investors highlighted in rela...
4983,q45a9a878195e,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 4, '2': 3, '1': 2, '0': 1, '3': 0}",What clarifications did analysts seek regardin...
4984,q2c57b56ff808,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 4, '1': 3, '2': 2, '0': 1, '3': 0}",How did Zoetis Inc.’s executives address regul...


In [ ]:
required_values.issubset(set([0,1,2,3,4]))

True

In [ ]:
# take only those rows for training in which we have the numbers 4,3,2,1,0 in values in qrel, because sometimes they just have 1 and 0s
required_values = set([0, 1, 2, 3, 4])
df_doc_dev_filtered = df_doc_dev[df_doc_dev['qrel'].apply(lambda x: required_values.issubset(set(x.values())))]

print("Filtered DataFrame head:")
display(df_doc_dev_filtered.head())
print(f"Number of rows before filtering: {len(df_doc_dev)}")
print(f"Number of rows after filtering: {len(df_doc_dev_filtered)}")

Filtered DataFrame head:


,uuid,messages,qrel,question
5,qc640e7d1c938,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 4, '2': 3, '4': 2, '0': 1, '3': 0}",What did Agilent Technologies’ executives high...
19,q267221ef200d,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 4, '1': 3, '0': 2, '2': 1, '3': 0}",How does Apple view the evolving industry land...
22,q3edb8b692639,"[{'role': 'user', 'content': 'Rank the followi...","{'2': 4, '4': 3, '0': 2, '3': 1, '1': 0}",Provide the latest reported subscriber count f...
39,q8a3f9336b517,"[{'role': 'user', 'content': 'Rank the followi...","{'2': 4, '3': 3, '4': 2, '1': 1, '0': 0}",Detail Apple's Cost of Goods Sold (COGS) for t...
40,q4947c1b05634,"[{'role': 'user', 'content': 'Rank the followi...","{'2': 4, '1': 3, '4': 2, '3': 1, '0': 0}",What is the current penetration rate of Apple ...


Number of rows before filtering: 4986
Number of rows after filtering: 3943


In [ ]:
# get those rows in full dev which exclude those in filtered

df_doc_dev_excluded = df_doc_dev[~df_doc_dev.index.isin(df_doc_dev_filtered.index)]
display(df_doc_dev_excluded.head())
print(f"Number of rows excluded: {len(df_doc_dev_excluded)}")

,uuid,messages,qrel,question
0,qe100cdf8e8f5,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}",How has Agilent Technologies’ instrument relia...
1,q723817294fbe,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '0': 2, '1': 1, '3': 1, '2': 0}",How did analysts question the outlook for Agil...
2,qd79970afaa57,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '1': 2, '0': 1, '2': 1, '3': 0}",How do sustainability or ESG considerations in...
3,qeb144d0e9991,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 3, '2': 2, '4': 1, '0': 0, '3': 0}","What exposure does Agilent Technologies, Inc. ..."
4,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '2': 2, '1': 1, '0': 0, '3': 0}",What sentiment trends are visible around Agile...


Number of rows excluded: 1043


In [ ]:
file_path = '/content/document_ranking_kaggle_dev.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_doc_dev = pd.DataFrame(data)

required_values = set([0, 1, 2, 3, 4])
df_doc_dev = df_doc_dev[df_doc_dev['qrel'].apply(lambda x: required_values.issubset(set(x.values())))]

# Extract the question from df_doc_dev
df_doc_dev["question"] = df_doc_dev["messages"].apply(lambda x: x[0]["content"].split("Question: ")[1].split("\n\nDocument")[0])

prepared_data = []
for index, row in df_doc_dev.iterrows():
    question = row['question']
    qrel = row['qrel']
    messages = row['messages'][0]['content']

    # Extract document indices and their content from the messages string
    doc_pattern = r'\[Document Index (\d+)\]\s*([\s\S]*?)(?=\[Document Index|$)'
    doc_matches = re.findall(doc_pattern, messages.split("Document Types to rank:\n")[1].split("\n\nYour response must be")[0])

    documents = {int(idx): content.strip() for idx, content in doc_matches}

    # Create data points for each document
    for doc_index, relevance in qrel.items():
        doc_index = int(doc_index)
        if doc_index in documents:
            prepared_data.append({
                'question': question,
                'document': documents[doc_index],
                'relevance': relevance
            })

# Convert the list of dictionaries to a DataFrame
df_prepared_data = pd.DataFrame(prepared_data)

print("Prepared data for reranker training:")
display(df_prepared_data.head())

Prepared data for reranker training:


,question,document,relevance
0,What did Agilent Technologies’ executives high...,10-K,4
1,What did Agilent Technologies’ executives high...,10-Q,3
2,What did Agilent Technologies’ executives high...,Earnings,2
3,What did Agilent Technologies’ executives high...,DEF14A,1
4,What did Agilent Technologies’ executives high...,8-K,0


In [ ]:
df_prepared_data["question"][4]

'What did Agilent Technologies’ executives highlight as the main risks and opportunities facing Agilent Technologies going forward?'

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer
from datasets import Dataset
import torch

# Convert the prepared data DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(df_prepared_data)

# Load the Jina reranker model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("jinaai/jina-reranker-v2-base-multilingual", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained("jinaai/jina-reranker-v2-base-multilingual",
                                                           trust_remote_code=True,
                                                           num_labels=1) # num_labels=1 for regression-like relevance score

# Preprocess the dataset: tokenize text and rename label column
def preprocess_function(examples):
    # Tokenize the question and document pair
    return tokenizer(examples["question"], examples["document"], truncation=True, padding="max_length")

# Apply the preprocessing function
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Rename the 'relevance' column to 'labels'
tokenized_dataset = tokenized_dataset.rename_column("relevance", "labels")

# Remove original text columns
tokenized_dataset = tokenized_dataset.remove_columns(["question", "document"])


def convert_labels_to_bfloat16(examples):
    # Convert labels to bfloat16 when the data is accessed
    examples['labels'] = torch.tensor(examples['labels'], dtype=torch.bfloat16).tolist()
    return examples

# Apply the transform - this modifies how data is returned when accessed
tokenized_dataset = tokenized_dataset.with_transform(convert_labels_to_bfloat16)


# Define training arguments
training_args = TrainingArguments(
    output_dir="./reranker_results",  # Output directory
    num_train_epochs=3,               # Number of training epochs
    per_device_train_batch_size=8,    # Batch size for training
    warmup_steps=500,                 # Number of warmup steps for learning rate scheduler
    weight_decay=0.01,                # Strength of weight decay
    logging_dir="./reranker_logs",    # Directory for storing logs
    logging_steps=10,
    eval_strategy="no",      # Evaluate every epoch
    save_strategy="epoch",            # Save checkpoint every epoch
    # load_best_model_at_last_check=True, # Load the best model at the end of training
    report_to="none", # Disable reporting to services like Weights & Biases
    bf16=True,  # Use BFloat16 instead of fp16
    fp16=False
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    # eval_dataset= # You would typically have a separate evaluation dataset here
)

# Start training
print("Starting reranker model training...")
trainer.train()
print("Reranker model training complete.")

Map:   0%|          | 0/24930 [00:00<?, ? examples/s]

Starting reranker model training...


Step,Training Loss
10,24.176200
20,24.828400
30,22.642200
40,26.243800
50,22.310800
60,21.723100
70,19.750000
80,17.096100
90,16.445700
100,9.806400


Reranker model training complete.


## Model Evaluation

### Subtask:
Evaluate the trained reranker model on the document ranking evaluation set and obtain scores for each document.

In [ ]:
# Load the document ranking evaluation data
file_path_eval = '/content/document_ranking_kaggle_eval.jsonl'
data_eval = []
with open(file_path_eval, 'r') as f:
    for line in f:
        data_eval.append(json.loads(line))
df_doc_eval = pd.DataFrame(data_eval)

# Extract the question from df_doc_eval
df_doc_eval["question"] = df_doc_eval["messages"].apply(lambda x: x[0]["content"].split("Question: ")[1].split("\n\nDocument")[0])

# Prepare evaluation data in the same format as training data
eval_prepared_data = []
for index, row in df_doc_eval.iterrows():
    question = row['question']
    messages = row['messages'][0]['content']
    sample_id = row['_id']

    # Extract document indices and their content from the messages string
    doc_pattern = r'\[Document Index (\d+)\]\s*([\s\S]*?)(?=\[Document Index|$)'
    doc_matches = re.findall(doc_pattern, messages.split("Document Types to rank:\n")[1].split("\n\nYour response must be")[0])

    documents = {int(idx): content.strip() for idx, content in doc_matches}

    # Create data points for each document in the evaluation set
    for doc_index, document_text in documents.items():
        eval_prepared_data.append({
            'sample_id': sample_id,
            'question': question,
            'document': document_text,
            'document_index': doc_index
        })

df_eval_prepared_data = pd.DataFrame(eval_prepared_data)

print("Prepared evaluation data for reranker:")
display(df_eval_prepared_data.head())

Prepared evaluation data for reranker:


,sample_id,question,document,document_index
0,doc_q39d7b7,How has Salesforce’s subscription and support ...,DEF14A,0
1,doc_q39d7b7,How has Salesforce’s subscription and support ...,10-K,1
2,doc_q39d7b7,How has Salesforce’s subscription and support ...,10-Q,2
3,doc_q39d7b7,How has Salesforce’s subscription and support ...,8-K,3
4,doc_q39d7b7,How has Salesforce’s subscription and support ...,Earnings,4


In [ ]:
# Tokenize the evaluation dataset
eval_dataset = Dataset.from_pandas(df_eval_prepared_data)

# Apply the same preprocessing as training data
eval_tokenized_dataset = eval_dataset.map(preprocess_function, batched=True)

# Remove original text columns, keep sample_id and document_index
eval_tokenized_dataset = eval_tokenized_dataset.remove_columns(["question", "document"])

print("Tokenized evaluation dataset:")
display(eval_tokenized_dataset)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenized evaluation dataset:


Dataset({
    features: ['sample_id', 'document_index', 'input_ids', 'attention_mask'],
    num_rows: 1000
})

### Subtask:
Use the trained reranker model to predict relevance scores for the evaluation data.

In [ ]:
# Predict scores on the evaluation dataset
print("Predicting relevance scores on evaluation data...")
predictions = trainer.predict(eval_tokenized_dataset)
print("Prediction complete.")

# The predictions are in the 'predictions' attribute of the PredictionOutput object
predicted_scores = predictions.predictions.squeeze().tolist()

# Add predicted scores to the evaluation dataframe
df_eval_prepared_data['predicted_score'] = predicted_scores

print("Evaluation data with predicted scores:")
display(df_eval_prepared_data.head())

Predicting relevance scores on evaluation data...


Prediction complete.
Evaluation data with predicted scores:


,sample_id,question,document,document_index,predicted_score
0,doc_q39d7b7,How has Salesforce’s subscription and support ...,DEF14A,0,0.468750
1,doc_q39d7b7,How has Salesforce’s subscription and support ...,10-K,1,3.453125
2,doc_q39d7b7,How has Salesforce’s subscription and support ...,10-Q,2,3.234375
3,doc_q39d7b7,How has Salesforce’s subscription and support ...,8-K,3,1.460938
4,doc_q39d7b7,How has Salesforce’s subscription and support ...,Earnings,4,2.156250


### Subtask:
Rank the documents for each query in the evaluation set based on the predicted scores and format the results for submission.

In [ ]:
# Rank documents for each sample_id based on predicted scores
ranked_results = []

# Group by sample_id and sort documents by predicted score in descending order
for sample_id, group in df_eval_prepared_data.groupby('sample_id'):
    ranked_group = group.sort_values(by='predicted_score', ascending=False)

    # Take the top 5 document indices for submission
    top_5_indices = ranked_group['document_index'].tolist()[:5]

    # Add to results list
    for doc_index in top_5_indices:
        ranked_results.append({
            'sample_id': sample_id,
            'target_index': doc_index
        })

# Convert to DataFrame for easy viewing and saving
df_ranked_results = pd.DataFrame(ranked_results)

print("Ranked document results for submission:")
display(df_ranked_results.head())

Ranked document results for submission:


,sample_id,target_index
0,doc_q007973,1
1,doc_q007973,2
2,doc_q007973,4
3,doc_q007973,0
4,doc_q007973,3


### Subtask:
Save the ranked document results to a CSV file in the required submission format.

In [ ]:
# Save the ranked document results to a CSV file
df_ranked_results.to_csv('results_doc_jinafinetuned_0_5epoch.csv', index=False)

## Save trained model

### Subtask:
Add code to save the fine-tuned reranker model to your Hugging Face repository.


In [ ]:
from huggingface_hub import HfApi, notebook_login

# Log in to Hugging Face Hub
notebook_login()

# Define your repository ID
repo_id = "your_username/your_model_name"  # Replace with your desired username and model name

# Push the model and tokenizer to the Hugging Face Hub
trainer.push_to_hub(repo_id)

HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-68d44ef6-438fe707074539261fd6dedd;01db20cd-ec80-4d97-ac07-10bc8ac5b0af)

Invalid username or password.

**Reasoning**:
The previous command failed due to an unauthorized error when trying to push to the Hugging Face Hub. This indicates an issue with the authentication token. I need to ensure the correct token is being used for login. Since I cannot directly access user secrets, I will instruct the user to make sure the `HF_TOKEN` secret is correctly set in their Colab environment and then retry pushing the model to the hub.



# Training Reranker on reduced sample with ListNet custom loss

In [ ]:
import torch
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, losses
from datasets import Dataset
from transformers import TrainingArguments
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import json
import re
from torch.utils.data import DataLoader

In [ ]:
# 1. Load BGE Cross-Encoder Model
model = CrossEncoder(
    "microsoft/mpnet-base",
    num_labels=1,  # Single output for ranking score
    device='cuda' if torch.cuda.is_available() else 'cpu'
)


config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/532M [00:00<?, ?B/s]

Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at microsoft/mpnet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Load the document ranking dev set
file_path = '/content/document_ranking_kaggle_dev.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_doc_dev = pd.DataFrame(data)

# Filter rows where qrel values include 0, 1, 2, 3, and 4
# required_values = set([0, 1, 2, 3, 4])
# df_doc_dev = df_doc_dev[df_doc_dev['qrel'].apply(lambda x: required_values.issubset(set(x.values())))]

# Extract the question from df_doc_dev
df_doc_dev["question"] = df_doc_dev["messages"].apply(lambda x: x[0]["content"].split("Question: ")[1].split("\n\nDocument")[0])


# 2. Prepare Training Data
def create_training_data(df_doc_dev):
    """
    Create training dataset with queries and document type rankings
    Format: Each query has all 5 document types with relevance scores [4,3,2,1,0]
    """

    training_examples = []
    name_to_docno = {"DEF14A": "0", "10-K": "1", "10-Q": "2", "8-K": "3", "Earnings": "4"}
    docno_to_name = {"0": "DEF14A", "1": "10-K", "2": "10-Q", "3": "8-K", "4": "Earnings"}

    for index, row in df_doc_dev.iterrows():
        query = row['question']
        qrel = row['qrel']
        messages = row['messages'][0]['content']

        docs = [docno_to_name[x] for x in qrel.keys()]
        labels = list(qrel.values())

        training_examples.append({
            "query": query,
            "docs": docs,
            "labels": labels
        })

    return training_examples

In [ ]:
# 3. Format Data for ListNet Loss
def format_data_for_listnet(training_examples):
    """
    Convert training examples to format required by ListNet loss
    """
    formatted_data = {
        "query": [],
        "docs": [],
        "labels": []
    }

    for example in training_examples:
        formatted_data["query"].append(example["query"])
        formatted_data["docs"].append(example["docs"])
        formatted_data["labels"].append(example["labels"])

    return formatted_data

# 4. Create Dataset
training_examples = create_training_data(df_doc_dev)
formatted_data = format_data_for_listnet(training_examples)

In [ ]:
# Split into train/test
train_data, test_data = train_test_split(
    list(zip(formatted_data["query"], formatted_data["docs"], formatted_data["labels"])),
    test_size=0.2,
    random_state=42,
)

# Split into train/valid
train_data, val_data = train_test_split(
    list(zip(formatted_data["query"], formatted_data["docs"], formatted_data["labels"])),
    test_size=0.2,
    random_state=42
)

In [ ]:
train_data

[('How has Keurig Dr Pepper’s beverage segment profitability trended over recent periods?',
  ['10-Q', '10-K', 'Earnings', '8-K', 'DEF14A'],
  [4, 3, 2, 1, 0]),
 ('How does management describe competitive advantages in generative AI developer tooling',
  ['Earnings', '10-K', 'DEF14A', '8-K', '10-Q'],
  [4, 3, 2, 1, 0]),
 ('What did Mohawk Industries’ leadership say about Mohawk Industries’ share repurchase plans?',
  ['10-K', '10-Q', 'Earnings', 'DEF14A', '8-K'],
  [2, 2, 1, 0, 0]),
 ('What investor views emerged on O’Reilly Automotive’s geographic expansion prospects within the United States?',
  ['10-K', 'DEF14A', 'Earnings', '10-Q', '8-K'],
  [4, 3, 2, 1, 0]),
 ('How did Tesla’s leadership address potential risks or uncertainties in forward-looking statements for Tesla?',
  ['DEF14A', '10-K', 'Earnings', '10-Q', '8-K'],
  [4, 3, 2, 1, 0]),
 ('How does Erie Indemnity Company view the pace of digital transformation in the insurance sector and its effect on market competitiveness?',
  

In [ ]:
# Create datasets
train_dataset = Dataset.from_dict({
    "query": [item[0] for item in train_data],
    "docs": [item[1] for item in train_data],
    "labels": [item[2] for item in train_data]
})

val_dataset = Dataset.from_dict({
    "query": [item[0] for item in val_data],
    "docs": [item[1] for item in val_data],
    "labels": [item[2] for item in val_data]
})

In [ ]:
# 5. Initialize ListNet Loss
loss_function = losses.ListNetLoss(
    model=model,
    activation_fn=torch.nn.Identity(),  # No activation since we want raw scores
    mini_batch_size=None  # Process full batches
)

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments

# 5. Use CrossEncoderTrainingArguments
training_args = CrossEncoderTrainingArguments(
    output_dir="./bge-reranker-sec-docs",
    num_train_epochs=3,
    per_device_train_batch_size=4,  # smaller batch size for ListNet
    per_device_eval_batch_size=4,
    warmup_steps=100,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=400,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_accumulation_steps=2,
    fp16=torch.cuda.is_available(),
    dataloader_drop_last=False,
    report_to=[]
)

In [ ]:
# 7. Initialize Trainer
trainer = CrossEncoderTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss_function,
)

# 8. Train the Model
print("Starting training...")
trainer.train()

Starting training...


Step,Training Loss,Validation Loss
200,1.436700,1.411904
400,1.371400,1.371460
600,1.368800,1.352065
800,1.345700,1.351549
1000,1.314300,1.340518
1200,1.334200,1.330576
1400,1.318300,1.336535


TrainOutput(global_step=1497, training_loss=1.3677224596898876, metrics={'train_runtime': 872.9645, 'train_samples_per_second': 13.705, 'train_steps_per_second': 1.715, 'total_flos': 0.0, 'train_loss': 1.3677224596898876, 'epoch': 3.0})

In [ ]:
# 9. Save the Fine-tuned Model
model.save("./fine-tuned-mpnet")
print("Model saved successfully!")

Model saved successfully!



Query: How has Salesforce’s subscription and support segment profitability trended over recent periods?
Document Type Rankings:
1. 10K: 0.7290
2. 10Q: 0.7215
3. Earnings: 0.5500
4. 8K: 0.2664
5. DEF14A: 0.2195
--------------------------------------------------

Query: How does Blackstone manage equity award burn rate or share pool availability?
Document Type Rankings:
1. 10K: 0.8041
2. DEF14A: 0.7911
3. 10Q: 0.7359
4. 8K: 0.2397
5. Earnings: 0.2180
--------------------------------------------------

Query: What questions were asked about A. O. Smith Corporation’s residential water heater market share metrics?
Document Type Rankings:
1. Earnings: 0.7006
2. 8K: 0.3919
3. 10K: 0.3458
4. DEF14A: 0.3315
5. 10Q: 0.3125
--------------------------------------------------

Query: How has the ratio of Corpay, Inc.’s recurring to one-time revenue evolved in the latest reporting period?
Document Type Rankings:
1. 10Q: 0.6790
2. 10K: 0.6766
3. 8K: 0.3607
4. Earnings: 0.3222
5. DEF14A: 0.2002
-----

In [ ]:
# yet to test: NDCG Evaluation
def calculate_ndcg_at_k(true_relevance, predicted_scores, k=5):
    """
    Calculate NDCG@K for evaluation
    """
    from sklearn.metrics import ndcg_score
    return ndcg_score([true_relevance], [predicted_scores], k=k)

# Example usage for NDCG evaluation
def evaluate_with_ndcg(model, test_data):
    """
    Evaluate model using NDCG metric
    """
    ndcg_scores = []

    for example in test_data:
        query = example["query"]
        doc_types = example["docs"]
        true_labels = example["labels"]

        # Get predictions
        pairs = [(query, doc_type) for doc_type in doc_types]
        predicted_scores = model.predict(pairs)

        # Calculate NDCG
        ndcg = calculate_ndcg_at_k(true_labels, predicted_scores)
        ndcg_scores.append(ndcg)

    return np.mean(ndcg_scores)

In [ ]:
# Now re-run training on whole of training set

In [ ]:
# Run final evaluations on eval set and sabe to submit

# 10. Evaluation Function
def evaluate_model(model, test_query, doc_types=["10K", "10Q", "8K", "Earnings", "DEF14A"]):
    """
    Evaluate the trained model on a test query
    """
    # Create query-document pairs
    pairs = [(test_query, doc_type) for doc_type in doc_types]

    # Get scores
    scores = model.predict(pairs)

    # Create ranking
    doc_scores = list(zip(doc_types, scores))
    doc_scores.sort(key=lambda x: x[1], reverse=True)

    print(f"\nQuery: {test_query}")
    print("Document Type Rankings:")
    for i, (doc_type, score) in enumerate(doc_scores, 1):
        print(f"{i}. {doc_type}: {score:.4f}")

    return doc_scores

# 11. Test the Model
if __name__ == "__main__":
    # Load the fine-tuned model for testing
    trained_model = CrossEncoder("./fine-tuned-mpnet")

    # Test queries
    # test_queries = [
    #     "What are the company's financial performance metrics?",
    #     "Are there any recent merger announcements?",
    #     "What is discussed in shareholder meetings?",
    #     "What were the latest quarterly results?"
    # ]

    for query in df_doc_eval["question"]:
        evaluate_model(trained_model, query)
        print("-" * 50)